# 🤖 Agentic Customer Care — Colab notebook

Self-contained version of the whole project (GreenThumb Marketplace customer-care agent): every class and function
is defined inline in the cells, no files and no Poetry needed. Run the sections top to bottom.

| # | Section |
|---|---------|
| 0 | Set up: env variables, utils, providers (embedding, SMTP) |
| 1 | RAG: ingestion, retrieval, evaluation (definition, run, test) |
| 2 | Agent: classes, middleware, tools, entity extractor, main workflow |
| 3 | API gateway |
| 4 | Run all steps |
| 5 | Documentation |

Each cell starts with a `# source:` comment pointing to the original file of the repo.

# 0. Set up

## 0.1 Dependencies, env variables, knowledge base

Add your secrets in Colab (🔑 *Secrets* panel): `OPENAI_API_KEY` (required) and optionally `OPENAI_API_BASE`,
`SMTP_USERNAME`, `SMTP_PASSWORD`, `GITHUB_TOKEN` (to clone a private repo).

*Installs the dependencies, pinned to the repo's `poetry.lock` versions.*

In [ ]:
import os, sys, subprocess

MARKER = "/tmp/.agentic_deps_installed"   # survives the runtime restart, so the install runs only once
PACKAGES = """langchain-core==1.6.10 langchain==1.4.4 langchain-community==0.4.2 langchain-openai==1.7.1
langchain-text-splitters langchain-classic==1.0.8 langgraph==1.2.14 langgraph-prebuilt==1.1.0 openai==3.3.0 pydantic==2.14.0
sentence-transformers==6.1.0 chromadb==1.5.9 langchain-huggingface==1.2.3 langchain-chroma==1.1.0 markdown2 bs4
fastapi==0.143.0 uvicorn==0.54.0 rank-bm25 ragas==0.4.3 python-dotenv nest_asyncio psutil""".split()

if not os.path.exists(MARKER):
    # Colab's preinstalled google-colab / google-adk pin older requests / opentelemetry than our dependencies need
    # (fastapi, langchain-community, ragas): pip prints a conflict warning for packages we do not use, so only real failures are shown.
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
        raise RuntimeError("pip install failed")
    open(MARKER, "w").close()
    if "google.colab" in sys.modules:
        # Colab has opentelemetry & co. already imported: restart so the new versions are loaded cleanly
        print("Dependencies installed. Restarting the runtime: run the next cells normally.")
        os.kill(os.getpid(), 9)
print("Dependencies ready.")


: 

*Sets the project folder and the environment variables (same as `.env.example`).*

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/Agentic-Customer-Care"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)  # logs are written to ./log

def _secret(name, required=False):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        if required:
            import getpass
            return getpass.getpass(f"{name}: ")
        return ""

# Same variables as .env.example
ENV = {
    "OPENAI_API_KEY": _secret("OPENAI_API_KEY", required=True),
    "OPENAI_API_BASE": _secret("OPENAI_API_BASE") or "",   # empty = official OpenAI endpoint
    "LLM_PROVIDER": "openai",
    "LLM_MODEL": "openai/gpt-oss-120b",
    "EXTRACTOR_LLM_PROVIDER": "openai",
    "EXTRACTOR_LLM_MODEL": "openai/gpt-oss-120b",
    "EMBEDDING_PROVIDER": "huggingface",     # "huggingface" or "openai"
    "EMBEDDING_MODEL_NAME": "",              # empty = provider default
    "SMTP_PROVIDER": "mailtrap",
    "SMTP_SERVER": "smtp.mailtrap.io",
    "SMTP_PORT": "2525",
    "SMTP_USERNAME": _secret("SMTP_USERNAME") or "",
    "SMTP_PASSWORD": _secret("SMTP_PASSWORD") or "",
    "SMTP_SENDER": "no-reply@yourdomain.com",
    "AGENT_GATEWAY_HOST": "127.0.0.1",
    "AGENT_GATEWAY_PORT": "8100",
    "AGENT_VERBOSE": "false",
}
os.environ.update(ENV)
print({k: ("***" if "KEY" in k or "PASSWORD" in k else v) for k, v in ENV.items()})

### Knowledge base

The agent needs `knowledge_base/` (documents, orders, products). Choose **one** source:
`git` (clone the repo; set `REPO_URL`, plus the `GITHUB_TOKEN` secret if private), `upload` (zip of the `knowledge_base` folder) or `drive`.

*Fetches the `knowledge_base` folder from the chosen source.*

In [ ]:
KB_SOURCE = "git"   # "git" | "upload" | "drive"
REPO_URL = "https://github.com/gmonitillodev-hub/Agentic-Customer-Care.git"
DRIVE_KB_PATH = "/content/drive/MyDrive/knowledge_base"

kb = Path(PROJECT_ROOT) / "knowledge_base"
if not (kb / "documents").exists():
    if KB_SOURCE == "git":
        token = _secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        !rm -rf /tmp/kb_repo && git clone -q --depth 1 {url} /tmp/kb_repo
        !cp -r /tmp/kb_repo/knowledge_base {PROJECT_ROOT}/
        # drop the pre-built vector store and chunks: section 1.3 rebuilds them with the chosen embeddings
        !rm -rf {PROJECT_ROOT}/knowledge_base/vectorstore {PROJECT_ROOT}/knowledge_base/ingestion
    elif KB_SOURCE == "upload":
        from google.colab import files
        up = files.upload()
        !unzip -q -o {list(up)[0]} -d {PROJECT_ROOT}/
    elif KB_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        !cp -r {DRIVE_KB_PATH} {PROJECT_ROOT}/knowledge_base
print(sorted(p.name for p in kb.iterdir()), "| documents:", len(list((kb / "documents").rglob("*.md"))))

## 0.2 All utils

*Helpers: `get_project_root` and `_merge_models` (deep merge of Pydantic models).*

In [ ]:
# source: src/utils/functions.py (get_project_root adapted: there is no repo/.git in the notebook)
from pathlib import Path
from pydantic import BaseModel

def get_project_root() -> Path:
    return Path(PROJECT_ROOT)

def _merge_models[T: BaseModel](current: T, new: T) -> T:
    merged: dict = {}
    for field in type(current).model_fields:
        old_val, new_val = getattr(current, field), getattr(new, field)
        if isinstance(old_val, BaseModel) and isinstance(new_val, BaseModel):
            merged[field] = _merge_models(old_val, new_val)
        else:
            merged[field] = old_val if new_val is None else new_val
    return type(current).model_validate(merged)

*Constants: agent and extractor prompts, escalation email template, memory token limit.*

In [ ]:
# source: src/utils/constants.py
CC_AGENT_SYSTEM_PROMPT = """
You are a customer care agent specialized in garden stores.

Output protocol (CRITICAL):
- Every step of yours MUST be a tool call. NEVER write plain text outside a tool call.
- To talk to the user you MUST call the `AgentResponse` tool. It is the ONLY way to send a message.
  Greetings, small talk, clarifying questions (e.g. missing order id), refusals, "I don't know" and final answers all go through `AgentResponse`.
- Data tools (knowledge base, orders, context, escalation) are used first, only when needed. Then call `AgentResponse` exactly once with the final answer.
- If the user message needs no data (e.g. "hello"), call `AgentResponse` directly, without other tools.
- Reply in the same language of the user.
- Tool arguments MUST be valid JSON. Never escape apostrophes (write l'ordine, NOT l\'ordine) and avoid double quotes and backslashes inside `answer`.

Rules:
- ALWAYS use the data tools to retrieve any data, information, or answer about products, orders, policies and gardening.
- NEVER use your internal knowledge or training data to answer questions.
- NEVER invent or assume information.
- Always specialize request adding context informations.
- If the tools do not provide an answer, clearly state that the requested informations are unknown.
- If a required information is missing (e.g. the order id), ask the user for it through `AgentResponse`, do not guess.
- Your answers must be concise, summarized in a few lines, and focused on the user’s request in a formal and polite way.
- Do not elaborate or add extra information not supported by the tool results.
- Always prioritize reliability and up-to-date information coming from the tools.
- If the user insert an orderId, you have to follow the requested format. Eg. user insert or-003 / 003, replace with the proper format "ORD-003"

Your goal:
  Provide only accurate, tool-based answers, always delivered through `AgentResponse`.

"""

CC_AGENT_EXTRACTOR_PROMPT = """

You are an agent specialized in entity extraction from messages of an LLM conversation.

Extract all user personal information entities present in the following conversation text and tools responses.
Pay attention to Human and Agent responses, but also to Tool that could returns relevant informations.

Output protocol (CRITICAL):
- You MUST answer ONLY by calling the `SessionEntityData` tool. NEVER write plain text or raw JSON in the message:
  the result is valid only if it is passed as the arguments of the tool call.
- Call the tool exactly once, even if nothing is found (in that case leave all the fields null).
- Use EXACTLY the field names of the tool. Do not rename or add fields.

Entities to extract and the tool fields to fill:

- user_info.name -> user first name
- user_info.surname -> user last name
- user_info.email -> user email
- user_info.address_info -> city, postal_code, region, street
- order_data -> dict with key the orderId (format "ORD-XXX") and value:
    - additional_info: additional relevant informations about the order
    - issues_to_be_solved: list of open issues
    - issues_resolved: list of solved issues
    - request_status: status of the user request for that order

Rules:
- Do not invent data or add extra text. Use null for every unknown value (never empty strings, never "unknown").
- Extract only information about the user, not about the store or the agent.
- Tool arguments MUST be valid JSON: no escaped apostrophes (write l'ordine, NOT l\'ordine), no trailing text.

"""

ESCALATION_EMAIL_TEMPLATE = """<!DOCTYPE html>
<html>
  <body style="font-family: Arial, sans-serif; color: #222;">
    <h2>Escalated request</h2>
    <p>A customer request needs human assistance:</p>
    <blockquote style="border-left: 3px solid #ccc; margin: 0; padding-left: 12px;">
      {message}
    </blockquote>
    <p style="color: #888; font-size: 12px;">— Support Team</p>
  </body>
</html>"""

MAX_MEMORY_TOKEN = 2000


*`setup_logger`: file and optional console logging per area (agent, tools, rag).*

In [ ]:
# source: src/utils/logger.py
import logging
from pathlib import Path
from typing import Literal

LOG_DIR = Path("log")

LogType = Literal["agent", "tools", "rag"]


def setup_logger(log_type: LogType, enable_console=False) -> logging.Logger:
    """Create a logger writing to log/<log_type>/{info,error}.log."""
    logger = logging.getLogger(f"{log_type}_logger")
    if logger.handlers:
        return logger  # already configured, avoid duplicated handlers

    log_dir = LOG_DIR / log_type
    log_dir.mkdir(parents=True, exist_ok=True)
    logger.setLevel(logging.DEBUG)

    error_file_handler = logging.FileHandler(log_dir / "error.log")
    error_file_handler.setLevel(logging.ERROR)

    info_file_handler = logging.FileHandler(log_dir / "info.log")
    info_file_handler.setLevel(logging.DEBUG)

    formator = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    error_file_handler.setFormatter(formator)
    info_file_handler.setFormatter(formator)

    logger.addHandler(error_file_handler)
    logger.addHandler(info_file_handler)

    if enable_console:
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter('\n %(asctime)s - %(levelname)s - %(message)s \n'))
        logger.addHandler(console_handler)

    return logger

## 0.3 Providers

### 0.3.1 Embedding

*Embedding providers (HuggingFace, OpenAI) and their factory.*

In [ ]:
# source: src/agentic_customer_care/providers/embedding_model_provider/embedding_provider.py
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()


class OpenAIEmbeddingProvider:
    def __init__(self, model_name: str | None = None):
        self.model_name = model_name or "text-embedding-3-small"

    def get_embeddings_model(self):
        return OpenAIEmbeddings(model=self.model_name)
    
class HuggingFaceEmbeddingProvider:
    def __init__(self, model_name: str | None = None):
        self.model_name = model_name or "sentence-transformers/all-MiniLM-L6-v2"

    def get_embeddings_model(self):
        return HuggingFaceEmbeddings(model_name=self.model_name)

class EmbeddingProviderFactory:
    @staticmethod
    def get_embedding_provider(provider_type: str | None = None, model_name: str | None = None):
        provider_type = provider_type or os.getenv("EMBEDDING_PROVIDER", "huggingface")
        model_name = model_name or os.getenv("EMBEDDING_MODEL_NAME") or None
        if provider_type == "openai":
            return OpenAIEmbeddingProvider(model_name)
        elif provider_type == "huggingface":
            return HuggingFaceEmbeddingProvider(model_name)
        else:
            raise ValueError(f"Unsupported provider: {provider_type}. Supported types are 'openai' and 'huggingface'.")

### 0.3.2 SMTP

*SMTP provider (Mailtrap) and its factory, used for human escalation.*

In [ ]:
# source: src/agentic_customer_care/providers/smtp/smtp_factory.py
import os
import smtplib
from email.message import EmailMessage
from dotenv import load_dotenv

class MailtrapSmtpProvider:
    
    def __init__(self):
        # Initialize the Mailtrap SMTP provider with necessary configurations
        load_dotenv()  # Load environment variables from .env file
        self.smtp_server = os.getenv("SMTP_SERVER", "smtp.mailtrap.io")
        self.smtp_port = int(os.getenv("SMTP_PORT", 2525))
        self.username = os.getenv("SMTP_USERNAME")
        self.password = os.getenv("SMTP_PASSWORD")
        self.sender = os.getenv("SMTP_SENDER", "no-reply@yourdomain.com")

    def send_email(self, to_email: str, subject: str, body: str, html_body: str | None = None):
        msg = EmailMessage()
        msg["Subject"] = subject
        msg["From"] = self.sender
        msg["To"] = to_email
        msg.set_content(body)
        if html_body:
            msg.add_alternative(html_body, subtype="html")

        try:
            with smtplib.SMTP(self.smtp_server, self.smtp_port) as server:
                server.starttls()
                server.login(self.username, self.password)
                server.send_message(msg)
        except Exception as e:
            raise ValueError(f"Failed to send email: {e}") from e


class SmtpFactory:
    @staticmethod
    def init_smtp_provider(provider_name: str = os.getenv("SMTP_PROVIDER", "mailtrap")):
        if provider_name == "mailtrap":
            return MailtrapSmtpProvider()
        else:
            raise ValueError(f"Unknown SMTP provider: {provider_name}")

Smoke test of the providers (no email is sent):

In [ ]:
emb = EmbeddingProviderFactory.get_embedding_provider().get_embeddings_model()
print("Embedding dim:", len(emb.embed_query("hello garden")))
print("SMTP provider:", type(SmtpFactory.init_smtp_provider()).__name__)

# 1. RAG

## 1.1 Ingestion definition

*Ingestion pipeline: extract, clean, chunk and index the documents into Chroma.*

In [ ]:
# source: src/agentic_customer_care/rag/ingestion.py
import os
from pathlib import Path


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
import markdown2
import json
from bs4 import BeautifulSoup

import re

rag_logger = setup_logger("rag")

def extract_documents_from_kb() -> list[Document]:
    """
    Extract documents from the knowledge base folder and return them as a list of Document objects.

    Returns:
        list[Document]: A list of Document objects containing the content and metadata of each document.
        
    Raises:
        FileNotFoundError: If the knowledge base folder does not exist or is not a directory
        
    """
    
    kd_folder = Path(get_project_root() / "knowledge_base" / "documents")
    
    print(f"\n    Extracting documents from knowledge base folder: {kd_folder}")
    
    documents = []
    
    if not kd_folder.exists() or not kd_folder.is_dir():
        rag_logger.error(f"Knowledge base folder not found: {kd_folder}")
        raise FileNotFoundError(f"Knowledge base folder not found: {kd_folder}")
    
    for topic_folder in kd_folder.iterdir():
        if topic_folder.is_dir():
            for document_file in topic_folder.iterdir():
                if document_file.is_file():
                    with open(document_file, "r", encoding="utf-8") as f:
                        content = f.read()
                        splitted_topic = (topic_folder.name).split("/")
                        topic = (splitted_topic)[len(splitted_topic) - 1]
                        documents.append(
                            Document(
                                page_content=content,
                                metadata={
                                    "topic": str(topic), 
                                    "source": str(document_file)
                                    }
                                )
                            )
    
    print(f"    Number of documents extracted from knowledge base: {len(documents)}")
    rag_logger.info(f"Extracted {len(documents)} documents from {kd_folder}")
    
    return documents
     
def preprocess_documents(documents: list[Document]) -> list[Document]:
    """
    Preprocess the documents by cleaning and normalizing the text.

    Args:
        documents (list[Document]): A list of Document objects to be preprocessed.
    Returns:
        list[Document]: A list of preprocessed Document objects.
    """
    
    print(f"\n    Preprocessing {len(documents)} documents...")
    
    for doc in documents:
        
        """ Convert markdown to plain text and clean the content """
        html = markdown2.markdown(doc.page_content)

        soup = BeautifulSoup(html, "html.parser")
        doc.page_content = soup.get_text(separator="\n")
        
        # Remove extra whitespace and normalize the text
        doc.page_content = re.sub(r'\s+', ' ', doc.page_content).strip()
        
        # Remove any unwanted characters or patterns (e.g., special characters, HTML tags)
        doc.page_content = re.sub(r'<[^>]+>', '', doc.page_content)
        
        # Remove markdown titles (#, ##, ecc.)
        doc.page_content = re.sub(r'^#+\s*', '', doc.page_content, flags=re.MULTILINE)
        
    print(f"    Preprocessed documents: {len(documents)}")
        
    return documents
                          
def chunk_documents(documents: list[Document]) -> list[Document]:
    """
    Chunk the documents into smaller pieces for better processing.

    Args:
        documents (list[Document]): A list of Document objects to be chunked.
    """
    print(f"\n    Chunking {len(documents)} documents with RecursiveCharacterTextSplitter...")
    
    splitter = RecursiveCharacterTextSplitter(
           chunk_size=1000,        
           chunk_overlap=200,     
           separators=["\n\n", "\n", " ", ""],
        )
    chunks = splitter.split_documents(documents)
    
    output_dir = Path(get_project_root() / "knowledge_base" / "ingestion")
    os.makedirs(output_dir, exist_ok=True)  # crea la cartella se non esiste

    output_path = os.path.join(output_dir, "chunks.jsonl")
    with open(output_path, "w", encoding="utf-8") as f:
        for doc in chunks:
            d = {
            "page_content": doc.page_content,
            "metadata": doc.metadata
            }
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
    
    print(f"    Number of chunks created: {len(chunks)}")
    rag_logger.info(f"Created {len(chunks)} chunks from {len(documents)} documents, saved to {output_path}")
    return chunks
    
def create_vectorstore_from_chunks(chunks: list[Document]):
    
    """
    Create a vector store from the given chunks of documents using embeddings.
    
    Args:
        chunks (list[Document]): A list of Document objects to be indexed in the vector store.

    """
    
    PROVIDER_TYPE = os.getenv("EMBEDDING_PROVIDER", "huggingface")  # set in .env
    
    print(f"\n    Creating vectorstore from {len(chunks)} chunks using {PROVIDER_TYPE} embeddings... \n")
    
    embedding_provider = EmbeddingProviderFactory.get_embedding_provider(provider_type=PROVIDER_TYPE)

    clear_vectorstore = input(f"    Do you want to clear the existing vectorstore? (y/n): ")
    if clear_vectorstore.lower() == "y":
        vectorstore = Chroma(persist_directory=str(get_project_root() / "knowledge_base" / "vectorstore"), 
                         embedding_function=embedding_provider.get_embeddings_model())
        
        if vectorstore._collection.count() > 0:
            vectorstore.delete_collection()
            print(" \n   Existing vectorstore cleared. \n")
        
    try:
        # Embedding + Indexing — tutto in una riga
        vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=embedding_provider.get_embeddings_model(),
                persist_directory=str(get_project_root() / "knowledge_base" / "vectorstore")
            )
        indexed_vector_number = vectorstore._collection.count()  # Trigger indexing
        
        if indexed_vector_number <= 0:
            raise ValueError("No vectors were indexed. Please check the input documents and embeddings.")
        
        print(f"\n  Vectorstore created and persisted with {indexed_vector_number} vectors.")
        rag_logger.info(f"Vectorstore created with {indexed_vector_number} vectors ({PROVIDER_TYPE} embeddings)")
        
    except Exception as e:
        print(f"Error creating vectorstore: {e}")
        rag_logger.error(f"Error creating vectorstore: {e}")
        raise e
    
def run_ingestion_pipeline():
    """
    Run the ingestion pipeline to extract, preprocess, and chunk documents from the knowledge base.
    
    Steps:
        1. Extract documents from the knowledge base folder.
        2. Preprocess the extracted documents.
        3. Chunk the preprocessed documents into smaller pieces.
        4. Create a vector store from the chunks using embeddings.
        
    """
    
    print("\n---------------------------------------------------------------")
    print("------------------------ Ingestion Pipeline ------------------- \n")
    print(" \n  Running ingestion pipeline...")
    rag_logger.info("Ingestion pipeline started")
    
    extracted_documents = extract_documents_from_kb()
    documents = preprocess_documents(extracted_documents)
    
    chunks = chunk_documents(documents)    
    create_vectorstore_from_chunks(chunks)
    
    print("\n   Ingestion pipeline completed successfully.")
    rag_logger.info("Ingestion pipeline completed")
    print("---------------------------------------------------------------\n")

## 1.2 Retrieval definition

*`HybridRetriever`: BM25 + vector search, loaded once as a singleton.*

In [ ]:
# source: src/agentic_customer_care/rag/retrival.py
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
import json
import threading
from langchain_core.documents import Document

rag_logger = setup_logger("rag")


class HybridRetriever:
    """
        Singleton: embeddings, Chroma and chunks are loaded once and reused.
        This avoid wrights loading for each retrieve improving the TTFT.
        
        Singleton because will be used in a tool, so We are confident that will
        not created more instances.
    """

    _instance: "HybridRetriever | None" = None
    _lock = threading.Lock()

    def __new__(cls):
        with cls._lock:
            if cls._instance is None:
                instance = super().__new__(cls)
                instance._initialized = False
                cls._instance = instance
        return cls._instance

    def __init__(self):
        with self._lock:
            if self._initialized:
                return
            rag_logger.info("Initializing HybridRetriever (loading embeddings, vectorstore, chunks)")
            embedding_provider = EmbeddingProviderFactory.get_embedding_provider()
            self._vectorstore = Chroma(
                persist_directory=str(get_project_root() / "knowledge_base" / "vectorstore"),
                embedding_function=embedding_provider.get_embeddings_model(),
            )
            chunks_path = Path(get_project_root() / "knowledge_base" / "ingestion" / "chunks.jsonl")
            self._chunks = []
            with open(chunks_path, "r", encoding="utf-8") as f:
                for line in f:
                    d = json.loads(line)
                    self._chunks.append(Document(page_content=d["page_content"], metadata=d["metadata"]))
            self._initialized = True
            rag_logger.info(f"HybridRetriever initialized - {len(self._chunks)} chunks loaded")
            
            

    def retrieve(self, query: str, top_k: int = 5, semantic_ratio: float = 0.6) -> list[Document]:
        
        if not (0.0 <= semantic_ratio <= 1.0):
            rag_logger.error(f"Invalid semantic_ratio: {semantic_ratio}")
            raise ValueError("Il parametro semantic_ratio deve essere compreso tra 0 e 1")
    
    # Calcolo automatico del peso per il BM25
        bm25_weight = 1.0 - semantic_ratio
        rag_logger.debug(f"Retrieve - query: {query} | top_k: {top_k} | semantic_ratio: {semantic_ratio}")
        bm25 = BM25Retriever.from_documents(self._chunks, k=top_k)
        semantic = self._vectorstore.as_retriever(search_kwargs={"k": top_k})
        # Hybrid — 40% BM25, 60% vector
        hybrid = EnsembleRetriever(retrievers=[bm25, semantic], weights=[bm25_weight, semantic_ratio])
        
        results = hybrid.invoke(query)[:top_k]
        rag_logger.info(f"Retrieved {len(results)} documents for query: {query}")
        return results



def test_retrieve_documents_from_vectorstore():
    
    TOP_K = 5
    
    proceed = True
    while proceed:
        
    
        query = input(" Enter a query to retrieve documents from the vectorstore: ")
        retrieved_docs = HybridRetriever().retrieve(query, TOP_K)
        
        print(f"\n    Retrieved {len(retrieved_docs)} documents for query: '{query}'")
        for i, doc in enumerate(retrieved_docs):
            print(f"        Document {i+1}: {doc}... \n\n")  # Print first 200 characters of each document
            
        user_input = input("    Do you want to test the document retrieval from the vectorstore? (y/n): \n")
        if user_input.lower() == "y":
            proceed = True
            
        elif user_input.lower() == "n":
            proceed = False
            print(" Skipping document retrieval test.")
            break
        else:
            print(" Invalid input. Please enter 'y' or 'n'.")

## 1.3 Run ingestion

Reads `knowledge_base/documents`, cleans, chunks (1000/200) and indexes into Chroma.
When asked whether to clear the existing vectorstore, answer `y` (recommended if you changed embeddings).

*Runs the ingestion pipeline.*

In [ ]:
run_ingestion_pipeline()

## 1.4 Run retrieval test

*Retrieves documents for a sample query.*

In [ ]:
QUERY = "What is the return window for an item?"
TOP_K, SEMANTIC_RATIO = 5, 0.6   # 1.0 = fully semantic, 0.0 = fully BM25

docs = HybridRetriever().retrieve(QUERY, TOP_K, SEMANTIC_RATIO)
print(f"Retrieved {len(docs)} documents for: {QUERY!r}\n")
for i, d in enumerate(docs, 1):
    print(f"[{i}] {d.metadata.get('source')}\n    {d.page_content[:300]}...\n")
# Interactive variant: test_retrieve_documents_from_vectorstore()

## 1.5 Evaluation definition

Evaluation datasets (question + ground truth):

*Evaluation dataset (question + ground truth).*

In [ ]:
# source: src/agentic_customer_care/rag/evaluation/eval_dataset.py
EVAL_DATASET = [
    # FAQ
    {
        "question": "Which payment methods does GreenThumb accept?",
        "ground_truth": "GreenThumb accepts Visa, Mastercard, American Express and Discover cards, PayPal and PayPal Credit, "
                        "Apple Pay and Google Pay, GreenThumb Gift Cards, and Affirm (buy now, pay later) on orders over $50.",
    },
    {
        "question": "I just placed an order by mistake. Can I still cancel it myself?",
        "ground_truth": "Within 1 hour of ordering you can modify or cancel the order directly from your account. After 1 hour "
                        "you must contact customer support as soon as possible, because changes cannot be guaranteed once the "
                        "order enters fulfillment.",
    },
    {
        "question": "How many rewards points do I need for a $5 reward and when do the points expire?",
        "ground_truth": "You earn 1 point per $1 spent and 100 points equal a $5 reward applied to your next order. Points "
                        "expire after 12 months of account inactivity.",
    },
    {
        "question": "Can I pay with more than one gift card, and can I get my money back from a gift card?",
        "ground_truth": "Yes, up to 3 gift cards can be combined per order and any remaining balance stays on the card. Gift "
                        "cards are non-refundable and cannot be exchanged for cash, except where required by law.",
    },
    {
        "question": "I forgot my password, how long is the reset link valid?",
        "ground_truth": "Click \"Forgot Password\" on the login page and a reset link is sent to your registered email. The "
                        "link expires in 24 hours.",
    },
    {
        "question": "What are the customer support phone number and opening hours?",
        "ground_truth": "Call 1-800-555-GROW (4769), Monday to Friday 8 AM - 6 PM EST and Saturday 9 AM - 1 PM EST. "
                        "Email is support@greenthumb.com, usually answered within 24 hours.",
    },
    # POLICIES
    {
        "question": "Can I return a packet of seeds that I have already opened?",
        "ground_truth": "No. Opened seed packets are not eligible for return, while unopened packets can be returned within "
                        "30 days. If GreenThumb-branded seeds fail to germinate under proper conditions, contact support "
                        "because they are covered by a germination guarantee.",
    },
    {
        "question": "How long do refunds take and is there a restocking fee?",
        "ground_truth": "Refunds are processed within 5-7 business days after the returned item is received and inspected, "
                        "to the original payment method. A 15% restocking fee may apply to large or heavy items such as power "
                        "tools and bulk soil orders. Original shipping charges are non-refundable unless the return is due to "
                        "GreenThumb's error or a defective product.",
    },
    {
        "question": "My plant arrived dead because of shipping. What do I have to do?",
        "ground_truth": "Report the issue within 48 hours of delivery with clear photos of the damaged plant and of the "
                        "packaging, via plantcare@greenthumb.com or 1-800-555-GROW. You can choose a free replacement or a "
                        "full refund.",
    },
    {
        "question": "My plant got pests three weeks after delivery even though I followed the care card. Am I covered?",
        "ground_truth": "Yes, the 30-Day Health Guarantee covers disease, pest infestation or failure to thrive within 30 days "
                        "of delivery despite following care instructions. You get a one-time replacement or store credit. "
                        "You must describe the care provided and send photos; the horticulture team may offer care advice first.",
    },
    {
        "question": "Does the plant guarantee cover frost damage after the plant was delivered?",
        "ground_truth": "No. Environmental damage such as frost after delivery, pet damage, accidental breakage or herbicide "
                        "exposure is not covered, nor are neglect or improper care, natural dormancy or cosmetic issues.",
    },
    {
        "question": "What are the shipping options, delivery times and costs?",
        "ground_truth": "Standard: 5-7 business days, free over $50 otherwise $5.99. Expedited: 2-3 business days, $12.99. "
                        "Express: 1-2 business days, $24.99. Freight (oversized): 7-14 business days, calculated at checkout. "
                        "Processing typically takes 1-2 business days.",
    },
    {
        "question": "I ordered a live plant on Thursday. When will it ship?",
        "ground_truth": "Live plants ship Monday through Wednesday only. Orders containing live plants placed after "
                        "Wednesday at 2 PM EST ship the following Monday.",
    },
    {
        "question": "Do you deliver to Hawaii or outside the United States?",
        "ground_truth": "No international shipping is available. GreenThumb ships within the contiguous US (48 states). "
                        "Alaska, Hawaii and U.S. territories have limited availability and additional shipping charges may apply, and live plants "
                        "cannot be shipped there.",
    },
    {
        "question": "What is the warranty period of a battery-powered tool and its lithium-ion battery?",
        "ground_truth": "Power tools (electric/battery mowers, trimmers, tillers, blowers) have a 2-year warranty. "
                        "Lithium-ion batteries are covered for 2 years or 500 charge cycles, whichever comes first.",
    },
    {
        "question": "How do I file a warranty claim for a defective tool?",
        "ground_truth": "Contact warranty@greenthumb.com or 1-800-555-GROW with your order number, date of purchase and a "
                        "description of the defect with clear photos. The claim is reviewed within 3-5 business days. If "
                        "approved the item is replaced (preferred), repaired for free, or a store credit/refund is issued.",
    },
    # PLATFORM ERRORS
    {
        "question": "What does error ERR009 mean and how should it be handled?",
        "ground_truth": "ERR009 is \"Payment declined by bank\". Ask the user to verify the card details and available funds, "
                        "try a different payment method, or contact their bank.",
    },
    {
        "question": "What is the recommended solution for ERR001?",
        "ground_truth": "ERR001 is a payment gateway timeout. Check the payment gateway service status and credentials, retry "
                        "the transaction after a few minutes, and contact the payment provider if the problem persists.",
    },
    {
        "question": "A customer says their discount code is rejected at checkout. Which error is it and what to check?",
        "ground_truth": "It is ERR002, invalid coupon code. Verify the coupon's validity, expiration date and usage limits, "
                        "and inform the user if the coupon is not applicable to their cart.",
    },
    {
        "question": "The customer never received the order confirmation email. What should support do?",
        "ground_truth": "This is ERR008, order confirmation email not sent. Check the email service integration and error "
                        "logs, resend the confirmation manually if needed, and advise the user to check spam/junk folders.",
    },
    {
        "question": "The shopping cart shows different content after the user switches device. Which error is this?",
        "ground_truth": "ERR005, cart synchronization error. Check session and user token validity, make sure the cart state "
                        "is updated client-side and server-side, and prompt the user to refresh the page or log in again.",
    },
    # GARDEN INFO
    {
        "question": "My compost pile is slimy and smells bad. What went wrong and how do I fix it?",
        "ground_truth": "The pile is too wet or has too many greens, so it has become anaerobic. Add browns and turn the pile. "
                        "The ideal ratio is about 3 parts browns to 1 part greens by volume, moist like a wrung-out sponge.",
    },
    {
        "question": "If I start a compost pile in October, when will it be ready and how often should I turn it in cold weather?",
        "ground_truth": "A pile started in October can produce finished compost by April or May. In cold weather turn it less "
                        "often: once a month is enough, and keep burying kitchen scraps in the center.",
    },
    {
        "question": "What is the difference between bypass and anvil pruners?",
        "ground_truth": "Bypass pruners have two curved blades that slide past each other like scissors, making clean, precise "
                        "cuts best for live wood and green growth. Anvil pruners have one blade cutting down onto a flat "
                        "surface, better for dead wood but they can crush live stems.",
    },
    {
        "question": "I need to cut a 3-inch thick branch. Which tool should I use?",
        "ground_truth": "Use a pruning saw, which is for branches over 2 inches in diameter (folding, bow or curved-blade "
                        "saws). Hand pruners cut up to 3/4 inch and loppers 3/4 to 2 inches.",
    },
    {
        "question": "At what angle and how often should I sharpen my garden tools?",
        "ground_truth": "Sharpen at least twice per season, following the original bevel angle, typically 20-25 degrees. File "
                        "in one direction only (push strokes), use a flat mill file for shovels/hoes and a diamond file or "
                        "whetstone for pruners, then remove the burr on the back side.",
    },
    {
        "question": "What soil and watering does lavender need?",
        "ground_truth": "Lavender needs full sun (6-8 hours), well-drained slightly alkaline soil (pH 6.5-7.5) with lean soil, "
                        "no rich compost, and 12-18 inches spacing. It is drought-tolerant once established; overwatering is "
                        "the number one killer.",
    },
    {
        "question": "At what level should the graft union sit when planting a rose?",
        "ground_truth": "The graft union (the bulge near the base) should sit at or just above soil level. Dig a hole twice as "
                        "wide and just as deep as the root ball, mix compost into the soil, then apply 2-3 inches of mulch "
                        "away from the stem.",
    },
    {
        "question": "What size pot and which variety are best for growing strawberries on a balcony?",
        "ground_truth": "Use pots at least 10-12 inches wide and 8 inches deep with drainage holes (one plant per 10-inch pot). "
                        "Day-neutral varieties are ideal for containers; recommended are 'Albion', 'Seascape', 'Tristar' and "
                        "'Mara des Bois'.",
    },
    {
        "question": "How deep and how far apart should I sow sunflower seeds, and when?",
        "ground_truth": "Sow directly after all frost danger has passed and soil reaches 50°F (10°C), 1 inch deep and 6 inches "
                        "apart. Thin seedlings to 12-24 inches apart once they have two sets of true leaves.",
    },
    {
        "question": "When and how do I start tomatoes from seed indoors?",
        "ground_truth": "Start 6-8 weeks before the last expected frost. Sow seeds 1/4 inch deep, 1 inch apart in seed-starting "
                        "mix, keep at 70-80°F (21-27°C) for germination, and give 14-16 hours of light with grow lights 2-3 "
                        "inches above the seedlings.",
    },
    {
        "question": "How much of an indoor herb can I harvest at once, and which herb needs its own pot?",
        "ground_truth": "Start harvesting once the plant has 6-8 sets of leaves and never remove more than one-third of the "
                        "plant at a time. Mint is extremely vigorous and should always be grown in its own container.",
    },
    {
        "question": "Which irrigation method is the most water-efficient and how much does a basic kit cost?",
        "ground_truth": "Drip irrigation is the most water-efficient (90%+ efficiency) and a basic DIY kit costs $50-$200. "
                        "Its emitters can clog, especially with hard water.",
    },
    {
        "question": "How thick should a straw mulch layer be and why use straw instead of hay?",
        "ground_truth": "Apply straw 4-6 inches deep because it compresses; it suits vegetable gardens, strawberry beds and "
                        "garlic. Use straw, not hay, because hay contains weed seeds.",
    },
    {
        "question": "How often and how should I water a Phalaenopsis orchid?",
        "ground_truth": "Water once every 7-10 days (more in summer, less in winter) by placing the pot under lukewarm "
                        "running water for 15-30 seconds and letting it drain completely. Never leave it in standing water; "
                        "overwatering is the most common cause of orchid death.",
    },
    {
        "question": "What are the NPK values of bone meal and blood meal and what are they used for?",
        "ground_truth": "Bone meal is 3-15-0, a phosphorus source for root growth and flowering, ideal for bulbs and "
                        "transplants. Blood meal is 12-0-0, a fast-acting nitrogen source for leafy vegetables that can burn "
                        "plants if overused.",
    },
    {
        "question": "When should I start pepper seeds and at what temperature do they germinate?",
        "ground_truth": "Start indoors 8-10 weeks before the last frost. Germination temperature is 80-85°F (27-29°C), best "
                        "with a heat mat, and takes 7-21 days. Transplant when night temperatures stay above 55°F (13°C), "
                        "spacing plants 18-24 inches apart.",
    },
    {
        "question": "How hot is a habanero compared to a jalapeño in SHU?",
        "ground_truth": "Jalapeño has medium heat at 2,500-8,000 SHU and is very productive. Habanero is very hot at "
                        "100,000-350,000 SHU with a fruity flavor.",
    },
    {
        "question": "When and how should I plant garlic cloves?",
        "ground_truth": "Plant in fall (typically October-November), 4-6 weeks before the ground freezes. Plant cloves pointed "
                        "end up, 2 inches deep and 6 inches apart in rows 12 inches apart, then mulch with 4-6 inches of straw "
                        "or shredded leaves.",
    },
    {
        "question": "Why can't I use garden soil in pots, and which mix is right for succulents?",
        "ground_truth": "Garden soil compacts heavily in containers, drains poorly, and may contain weed seeds, pests and "
                        "diseases. Succulents need a cactus and succulent mix with extra perlite and sand for fast drainage.",
    },
    {
        "question": "What gardening tasks should I do in March according to the planting calendar?",
        "ground_truth": "Indoors start tomatoes, peppers, eggplant and herbs 6-8 weeks before last frost. Outdoors, if the soil "
                        "is workable, direct sow peas, spinach and fava beans and plant bare-root roses, trees and berry bushes. "
                        "Also turn the compost pile and prepare raised beds with fresh compost.",
    },
    {
        "question": "Which soil pH do blueberries prefer, and what happens when the pH is too high?",
        "ground_truth": "Blueberries and other acid-loving plants (azaleas, rhododendrons) prefer pH 4.5-5.5. At high (alkaline) "
                        "pH, iron, manganese and zinc become unavailable, causing chlorosis (yellowing leaves).",
    },
    {
        "question": "What planting depth and spacing do tulips, daffodils and crocus need?",
        "ground_truth": "Tulips: 6-8 inches deep, 4-6 inches apart. Daffodils: 6-8 inches deep, 6-8 inches apart. Crocus: 3-4 "
                        "inches deep, 2-3 inches apart. Plant September through November when soil is below 60°F (15°C).",
    },
    {
        "question": "What should I do in the garden once the last frost has passed in spring?",
        "ground_truth": "Transplant warm-season seedlings (tomatoes, peppers, squash, cucumbers), direct sow beans, corn, "
                        "sunflowers and zinnias, install drip lines and soaker hoses, apply 2-3 inches of fresh mulch and set "
                        "up supports like tomato cages and trellises.",
    },
    {
        "question": "How do I water a succulent and how can I tell I am overwatering it?",
        "ground_truth": "Use the soak and dry method: water thoroughly until it drains, then let the soil dry completely. "
                        "Water every 7-14 days in summer and every 3-4 weeks in winter. Mushy, translucent leaves signal "
                        "overwatering or root rot.",
    },
    {
        "question": "What is the best time of day to water in summer and how much water per week?",
        "ground_truth": "Water early in the morning (6-10 AM) and avoid midday. Most gardens need 1-1.5 inches of water per "
                        "week in summer, delivered deeply and infrequently rather than in shallow daily sprinklings.",
    },
    {
        "question": "What are the pros and cons of clay soil and which plants suit it?",
        "ground_truth": "Clay soil is rich in nutrients and retains moisture, but drains poorly, warms slowly in spring and "
                        "compacts easily. It suits roses, asters, fruit trees and many ornamental shrubs.",
    },
    {
        "question": "What are the ideal dimensions for a vegetable raised bed?",
        "ground_truth": "Width 3-4 feet so you can reach the center, length 6-8 feet is most common, and depth at least 10-12 "
                        "inches (18-24 inches for deep-rooted crops like carrots and tomatoes).",
    },
    {
        "question": "How can I protect my garden plants from a hard frost?",
        "ground_truth": "Hard frost is below 28°F (-2°C). Use frost blankets or row covers (4-8°F of protection), apply 4-6 "
                        "inches of mulch after the ground freezes, and water the ground thoroughly before the freeze. Never "
                        "put plastic directly on plants because it conducts cold and causes freeze damage.",
    },
    {
        "question": "At what temperature should tropical plants be moved indoors in autumn, and which cover crops can I sow?",
        "ground_truth": "Bring tropical plants indoors before nighttime temperatures dip below 50°F (10°C). Plant a cover crop "
                        "such as winter rye, crimson clover or field peas in empty vegetable beds to prevent erosion and add "
                        "nutrients.",
    },
]

*Evaluation dataset with 25 test cases (used by default).*

In [ ]:
# source: src/agentic_customer_care/rag/evaluation/eval_dataset_25.py
EVAL_DATASET_25 = [
    # FAQ
    {
        "question": "Do I need an account to place an order and what do I gain by creating one?",
        "ground_truth": "No, you can check out as a guest. A free account lets you track orders, save addresses, view order "
                        "history, earn rewards points, and manage returns more easily.",
    },
    {
        "question": "How can a guest track an order?",
        "ground_truth": "Guest orders can be tracked at greenthumb.com/track using the order number and email address.",
    },
    {
        "question": "Is my payment information secure and how is sales tax calculated?",
        "ground_truth": "Yes. All transactions are encrypted with 256-bit SSL, the site is PCI-DSS compliant and never stores full "
                        "credit card numbers. Sales tax is calculated based on the shipping destination, according to state and "
                        "local tax laws.",
    },
    # Policies
    {
        "question": "What is the return window and what conditions must an item meet to be returned?",
        "ground_truth": "The return window is 30 days from the date of delivery. Items must be unused, in original packaging and "
                        "in resalable condition, and proof of purchase (order confirmation email or order number) is required.",
    },
    {
        "question": "Can I return an opened bag of soil or fertilizer?",
        "ground_truth": "No. Only unopened bags of soil, mulch and fertilizer can be returned within 30 days; opened bags are not "
                        "eligible for return.",
    },
    {
        "question": "How long do I have to report a plant damaged in transit and what must I send?",
        "ground_truth": "You must report it within 48 hours of delivery, with clear photographs of the damaged plant and of the "
                        "packaging, by emailing plantcare@greenthumb.com or calling 1-800-555-GROW. You get a free replacement or "
                        "full refund, and you do not need to ship the plant back.",
    },
    {
        "question": "What is the warranty period for hand tools, cutting tools and irrigation equipment?",
        "ground_truth": "Hand tools such as pruners, trowels and rakes have a 5-year warranty, cutting tools such as loppers and "
                        "hedge shears have 3 years, and irrigation equipment such as timers, drip kits and hoses has 1 year.",
    },
    {
        "question": "What are the standard and express shipping costs and delivery times?",
        "ground_truth": "Standard shipping takes 5-7 business days and is free on orders over $50, otherwise $5.99. Express "
                        "shipping takes 1-2 business days and costs $24.99.",
    },
    # Platform errors
    {
        "question": "What is the cause and solution of ERR004?",
        "ground_truth": "ERR004 is a user authentication failure. Ensure the user is entering the correct credentials, offer "
                        "password reset options and check for server-side authentication issues.",
    },
    {
        "question": "A customer cannot ship the order because no delivery option is offered at checkout. Which error is it?",
        "ground_truth": "It is ERR007, shipping method unavailable. Review shipping zones and rates configuration, ensure the "
                        "carrier APIs are reachable and offer alternative shipping options if available.",
    },
    {
        "question": "A user was logged out in the middle of checkout. Which error is this and what should be done?",
        "ground_truth": "It is ERR010, session expired. Notify the user and prompt them to log in again, and extend the session "
                        "timeout settings if appropriate.",
    },
    # Garden info
    {
        "question": "When and where should I plant roses and how far apart?",
        "ground_truth": "Plant roses in early spring after the last frost or in fall (6 weeks before the first frost), in full "
                        "sun with at least 6 hours of direct sunlight, spaced 2-3 feet apart in rich, well-drained, slightly "
                        "acidic soil (pH 6.0-6.5).",
    },
    {
        "question": "What are the options to lower soil pH and how accurate is a home test kit?",
        "ground_truth": "To lower pH you can use elemental sulfur (slow but long-lasting, 1-2 lbs per 100 sq ft), iron sulfate "
                        "(faster but shorter), peat moss or pine needle mulch. Home test kits are accurate to about ±0.5 pH "
                        "units, good for general guidance.",
    },
    {
        "question": "What is the ideal ratio of browns to greens in a compost pile and what should I not compost?",
        "ground_truth": "Aim for about 3 parts browns to 1 part greens by volume. Do not compost meat, fish and bones, dairy, "
                        "cooking oils and grease, pet waste, diseased plants or weeds with seeds, or chemically treated wood or "
                        "plants.",
    },
    {
        "question": "Which organic fertilizer is gentle enough that it cannot burn seedlings?",
        "ground_truth": "Worm castings (1-0-0) give gentle, balanced nutrition, cannot burn plants and are perfect for seedlings "
                        "and containers.",
    },
    {
        "question": "What is a simple homemade all-purpose organic fertilizer blend?",
        "ground_truth": "Mix 4 parts seed meal (alfalfa or soybean), 1 part bone meal and 1 part kelp meal, and apply 4-6 cups "
                        "per 100 square feet of garden bed.",
    },
    {
        "question": "What lawn care tasks should I do in autumn?",
        "ground_truth": "Aerate compacted lawns to improve water and nutrient penetration, overseed bare patches with "
                        "cool-season grass seed and apply a fall fertilizer to strengthen roots before winter.",
    },
    {
        "question": "How much does mulch reduce evaporation and which mulch suits blueberries?",
        "ground_truth": "Mulch reduces evaporation by up to 70%. Pine needles, applied 2-3 inches deep, are slightly acidifying "
                        "and best for acid-loving plants such as blueberries, azaleas and rhododendrons.",
    },
    {
        "question": "What temperature and light do tomato seedlings need to germinate and grow indoors?",
        "ground_truth": "Keep the temperature between 70-80°F (21-27°C) for germination, and provide 14-16 hours of light per "
                        "day under grow lights kept 2-3 inches above the seedlings.",
    },
    {
        "question": "How should I transplant tomato seedlings outdoors?",
        "ground_truth": "Harden off the seedlings over 7-10 days by gradually increasing outdoor exposure, plant deeply by "
                        "burying up to two-thirds of the stem, and space plants 24-36 inches apart. Soil should be at least "
                        "60°F (15°C).",
    },
    {
        "question": "What humidity and potting medium do orchids need, and how often should I repot?",
        "ground_truth": "Orchids prefer 50-70% humidity and need orchid bark mix, never regular potting soil, which suffocates "
                        "the roots. Repot every 1-2 years or when the bark has decomposed and no longer drains well.",
    },
    {
        "question": "When is garlic ready to harvest and how should it be cured?",
        "ground_truth": "Harvest when the lower third of the leaves have turned brown, typically June-July, after stopping "
                        "watering 2-3 weeks earlier. Cure in a warm, dry, well-ventilated area for 2-3 weeks, trim roots and "
                        "stems, then store in a cool, dark place.",
    },
    {
        "question": "What is the difference between hardneck and softneck garlic?",
        "ground_truth": "Hardneck garlic produces scapes, has a stronger flavor and fewer but larger cloves, and suits cold "
                        "climates. Softneck has no scapes, a milder flavor, more cloves per bulb, stores longer, is ideal for "
                        "braiding and suits mild climates.",
    },
    {
        "question": "How should I prune roses and when?",
        "ground_truth": "Prune in late winter or early spring, removing dead or crossing branches, and cut at a 45° angle above "
                        "an outward-facing bud. Deadhead spent blooms to encourage reblooming.",
    },
    {
        "question": "What are the symptoms of overwatering an orchid and how do I water it properly?",
        "ground_truth": "Overwatering is the most common cause of orchid death; brown, mushy roots indicate rot. Water once "
                        "every 7-10 days by putting the pot under lukewarm running water for 15-30 seconds, let it drain "
                        "completely and never leave it in standing water.",
    },
]

RAGAS evaluator:

*`RagasEvaluator` and `evaluate_rag`: retrieval, answer generation and RAGAS metrics.*

In [ ]:
# source: src/agentic_customer_care/rag/evaluation/ragas_evauation.py
import os
import sys
import time
import types

from langchain.chat_models import init_chat_model

# ragas 0.4.3 imports a module removed in langchain-community 0.4; stub it (used only for isinstance checks)
if "langchain_community.chat_models.vertexai" not in sys.modules:
    _vertexai_stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _vertexai_stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _vertexai_stub

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

class RagasEvaluator():
    
    def __init__(self):
        
        self.retriever = HybridRetriever()
        self.model = None
        
        self.extraction_results: list = []
        self.dataset = EVAL_DATASET_25
                 
        self.model = init_chat_model(
            model=os.getenv("EXTRACTOR_LLM_MODEL","openai/gpt-oss-120b"),
            model_provider=os.getenv("EXTRACTOR_LLM_PROVIDER","openai"),
            temperature=0, 
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=os.getenv("OPENAI_API_BASE") or None
        )
        
    def extract_results(self, top_k, semantic_ratio):
        
        
        print(f"\n Evaluating rag with {len(self.dataset)} test cases")
        for test_case in self.dataset:
            
            test_result = self.retrieve_data_from_kb(test_case["question"], top_k, semantic_ratio)    
            self.extraction_results.append(
                {
                    "question": test_case["question"],
                    "answer": test_result["answer"],
                    "context": test_result["chunks"],
                    "ground_truth": test_case["ground_truth"]
                }
            )
            
        print(f"    Data retrieved from KB !!")

    def retrieve_data_from_kb(self, query, top_k, semantic_ratio):
                  
        t0 = time.time()

        docs_retrieved = self.retriever.retrieve(query,  top_k, semantic_ratio)
        latency_retrieval = time.time() - t0

        prompt_template = ChatPromptTemplate.from_template("""
            You are an expert assistant. Answer ONLY based on available documents.
            If the information is not prensent, return a direct response without generate random content
            
            DOCUMENTs:
            {context}

            QUESTION: {input}

            ANSWER:"""
        )
        
        document_chain = create_stuff_documents_chain(self.model, prompt_template)

        t1 = time.time()
        answer = document_chain.invoke({
            "input": query,
            "context": docs_retrieved,
        })
        latency_generation = time.time() - t1

        return {
            "query":              query,
            "chunks":             docs_retrieved,
            "answer":             answer,
            "latency_retrieval":  round(latency_retrieval, 3),
            "latency_generation": round(latency_generation, 3),
            "latency_total":      round(time.time() - t0, 3),
        }
        
    @staticmethod    
    def build_eval_dataset(results: list) -> EvaluationDataset:
        
        print(f"\n  Buiding eval dataset ...")
        samples = [
            SingleTurnSample(
                user_input=r["question"],
                response=r["answer"],
                retrieved_contexts=[doc.page_content for doc in r["context"]],
                reference=r["ground_truth"]
            )
            for r in results
        ]
        return EvaluationDataset(samples=samples)

        
    def ragas_executor(self):
        
        evaluator_llm = LangchainLLMWrapper(self.model)
        # Without explicit embeddings ragas falls back to OpenAI embeddings and the OPENAI_API_KEY
        evaluator_embeddings = LangchainEmbeddingsWrapper(
            EmbeddingProviderFactory.get_embedding_provider().get_embeddings_model()
        )
        
        dataset = self.build_eval_dataset(self.extraction_results)
        print(f"\n  Started RAGAS evaluation ...")
        scores = evaluate(
            dataset=dataset,
            metrics=[
                Faithfulness(),
                ResponseRelevancy(strictness=1),
                LLMContextRecall(),
            ],
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
        )
        return scores, len(self.dataset)
        
""" Faithfulness = 1.0000
La risposta generata dall’LLM risulta completamente supportata dai context recuperati. Non emergono segnali di hallucination sui casi valutati.

Answer Relevancy = 0.9738
La risposta è fortemente pertinente rispetto alla domanda utente e risponde in modo quasi completo al bisogno informativo.

Context Recall = 1.0000
I context recuperati contengono tutte le informazioni necessarie rispetto alla ground truth/reference. Il retriever sembra quindi recuperare correttamente le informazioni rilevanti. """
    
def evaluate_rag():
    print(f"\n-----------------------------------------------------")
    print("--------------- RAG Evaluator - RAGAS ---------------")
    print(f"-----------------------------------------------------")

    top_k = int(input("\n Insert the top_k value: "))
    print("\n  Semantic ratio is a value that defines the proportion \n between similarity and keyword search (1 full semantic) \n")
    semantic_ratio = float(input("    Insert the semantic ratio (from 0 to 1): "))
    
    evaluator = RagasEvaluator()   
    evaluator.extract_results(top_k, semantic_ratio)
    
    metrics, eval_cases = evaluator.ragas_executor()
    
    print(f"\n    -----------------------------------------------------")
    print(f"    ------------------ Evaluation ended -----------------")
    
    print("\n   Models specifications:")
    print(f"     - Extractor model -> {os.getenv("EXTRACTOR_LLM_MODEL","openai/gpt-oss-120b")}")
    print("\n   Retrival parameters:")
    print(f"     - top_k -> {top_k}")
    print(f"     - semantic_ratio -> {semantic_ratio}")
    print(f"    Number of evaluation cases: {eval_cases}")
    
    print(f"\n    Metrics:")
    print(f"        {metrics}")

## 1.6 Evaluation test

Runs the RAGAS evaluation (faithfulness, answer relevancy, context recall) with the `EXTRACTOR_LLM_*` model.
It makes many LLM calls (one retrieval + generation per case): set `N_CASES` for a quicker run.

*Runs the RAGAS evaluation on the dataset.*

In [ ]:
import nest_asyncio; nest_asyncio.apply()   # ragas runs an event loop internally

TOP_K, SEMANTIC_RATIO = 5, 0.6
N_CASES = None   # e.g. 5 for a quick run, None = full dataset

evaluator = RagasEvaluator()
if N_CASES:
    evaluator.dataset = evaluator.dataset[:N_CASES]
evaluator.extract_results(TOP_K, SEMANTIC_RATIO)
metrics, n_cases = evaluator.ragas_executor()
print(f"Cases: {n_cases} | top_k={TOP_K} | semantic_ratio={SEMANTIC_RATIO}")
print(metrics)
# Interactive variant (asks top_k and semantic_ratio): evaluate_rag()

# 2. Agent

## 2.1 Classes

*Agent response models.*

In [ ]:
# source: src/agentic_customer_care/agent/classes/cc_agent.py
from pydantic import BaseModel, Field

class AgentResponse(BaseModel):
    """Send the reply to the user. MUST be called to deliver every message to the user
    (greetings, clarifying questions, refusals and final answers), after any data tool needed."""
    answer: str = Field(description="The reply shown to the user, in the user's language, concise and formal")
    confidence: float = Field(description="A score from 0 to 10")
    sources: str = Field(description="Source of data used: KB, ConversationContext, Other")
    needs_humans: bool = Field(description="True if a human operator is needed")

class InteractResponse(BaseModel):
    data: AgentResponse
    sessionId: str


*Session entity models: user info, addresses, orders, with deep merge.*

In [ ]:
# source: src/agentic_customer_care/agent/classes/sessions.py
from pydantic import BaseModel, Field

class AddressInfo(BaseModel):
    city: str | None = None
    postal_code: str | None = None
    region: str | None = None
    street: str | None = None

class UserInfo(BaseModel):
    name: str | None = None
    surname: str | None = None
    email: str | None = None
    address_info: AddressInfo | None = None

class OrderData(BaseModel):
    additional_info: str
    issues_to_be_solved:list[str]
    issues_resolved:list[str]
    request_status: str    

class SessionEntityData(BaseModel):
    """Entities extracted from the conversation. Return them by calling this tool: it is the only valid way to answer."""
    user_info: UserInfo | None = None
    order_data: dict[str, OrderData] | None = Field(description="Dict with orderId -> OrderData that keep track of already known info", default=None)
    
class SessionEntity(BaseModel):
    sessionId: str
    data: SessionEntityData = Field(default_factory=SessionEntityData)

    def merge_data(self, new_data: SessionEntityData) -> None:
        """Deep-merge new_data into self.data; None values never overwrite existing ones."""
        self.data = _merge_models(self.data, new_data)


## 2.2 Middleware

*Middleware: prompt enrichment with known user data and background entity updater.*

In [ ]:
# source: src/agentic_customer_care/agent/middlewares/mw.py
from langchain.agents.middleware import AgentMiddleware, AgentState, ModelRequest, after_agent, dynamic_prompt
import asyncio
from langgraph.runtime import Runtime

agent_logger = setup_logger("agent")


def build_prompt_enrichment(system_prompt: str):

    @dynamic_prompt
    def entities_prompt(request: ModelRequest[SessionEntity]) -> str:
        """Build the full system prompt, appending known user entities from the runtime context."""
        ctx = request.runtime.context
        known = ctx.data.model_dump(exclude_none=True) if ctx else {}
        if not known:
            return system_prompt
        return f"{system_prompt}\n\nKnown user data (do not ask for it again): {known}"
    return entities_prompt

def build_entities_updater(extractor: "EntityExtractorAgent") -> AgentMiddleware:
    """After the agent run, extract entities in the background without delaying the answer.
    
    The hook schedules the extraction with asyncio.create_task (not awaited) and returns
    immediately, so ainvoke() completes and the answer is returned right away.
    Async-only: the agent must be run with ainvoke().
    """
    
    @after_agent
    async def update_entities(state: AgentState, runtime: Runtime[SessionEntity]) -> None:
        async def _extract() -> None:
            # extractor.invoke is sync and blocking: to_thread runs it in a worker thread so the
            # event loop stays free; the await only suspends this task, not the loop.
            # the task is not awaited, so exceptions would otherwise be silently lost
            try:
                extracted = await asyncio.to_thread(extractor.invoke, state)
                if extracted:
                    runtime.context.merge_data(extracted)
                    agent_logger.debug(f"Session entities updated -> {extracted}")
            except Exception as e:
                agent_logger.error(f"Background entity extraction failed: {e}")
    
        asyncio.create_task(_extract())
    return update_entities

## 2.3 Tools

*Order tools: `get_order_status` and `get_order_details`.*

In [ ]:
# source: src/agentic_customer_care/agent/tools/oms.py
from pathlib import Path
from langchain_core.tools import tool
import json

tools_logger = setup_logger("tools")


def load_orders_from_kb() -> list[dict]:
    """
    Load orders from the knowledge base folder and return them as a list of dictionaries.

    Returns:
        list[dict]: A list of dictionaries containing the content and metadata of each order.
    """
    orders_kb = Path(get_project_root() / "knowledge_base" / "orders") 
    
    if not orders_kb.exists():
        tools_logger.error(f"Orders knowledge base not found: {orders_kb}")
        raise ValueError("Orders knowledge base not found.")
    orders:list = []

    try:
        for order_file in orders_kb.iterdir():
            if order_file.is_file() and order_file.suffix == ".json":
                with open(order_file, "r", encoding="utf-8") as f:
                    order_data = json.load(f)
                    orders.extend(order_data.get("orders", []))
                    
    except Exception as e:
        tools_logger.error(f"Error loading orders from knowledge base: {e}")
        raise ValueError(f"Error loading orders from knowledge base: {e}")

    tools_logger.debug(f"Loaded {len(orders)} orders from knowledge base")
    return orders

@tool
def get_order_status(order_id: str) -> str:
    """
    Get the status of an order from the Order Management System (OMS).

    Args:
        order_id (str): The unique identifier of the order that must have the following format `ORD-XXX`.
    
    Returns:
        str: The status of the order.
        
    Raises:
        ValueError: If the order_id is empty or contains only whitespace or if the order is not found in the knowledge base.
    
    """
        
    tools_logger.info(f"Tool get_order_status called - order_id: {order_id}")
    if not order_id.strip():
        tools_logger.error("get_order_status: empty order_id")
        raise ValueError("Order ID cannot be empty or whitespace.")
    
    orders = load_orders_from_kb()

    requested_order = [order for order in orders if order.get("order_id") == order_id]
    if not requested_order:
        tools_logger.error(f"get_order_status: order not found - order_id: {order_id}")
        raise ValueError("Order not found.")

    return requested_order[0].get("status")

def get_order_details(order_id: str) -> dict:
    """
    Get the details of an order from the Order Management System (OMS).

    Args:
        order_id (str): The unique identifier of the order that must have the following format `ORD-XXX`..

    Returns:
        dict: A dictionary containing the details of the order.

    Raises:
        ValueError: If the order_id is empty or contains only whitespace or if the order is not found in the knowledge base.
    """
    tools_logger.info(f"Tool get_order_details called - order_id: {order_id}")
    if not order_id.strip():
        tools_logger.error("get_order_details: empty order_id")
        raise ValueError("Order ID cannot be empty or whitespace.")

    orders = load_orders_from_kb()

    requested_order = [order for order in orders if order.get("order_id") == order_id]
    if not requested_order:
        tools_logger.error(f"get_order_details: order not found - order_id: {order_id}")
        raise ValueError("Order not found.")

    return requested_order[0]

*RAG tool: `retrieve_documents_from_rag`.*

In [ ]:
# source: src/agentic_customer_care/agent/tools/rag.py
from langchain_core.documents import Document
from langchain_core.tools import tool 

tools_logger = setup_logger("tools")

@tool
def retrieve_documents_from_rag(query: str) -> list[Document]:
    """
    Retrieve documents from the vectorstore based on a query.
    Is needed a preprocess step to ensure the query is in English language before calling this function
    and contains only the relevant information to retrieve documents from the vectorstore. 
    
    Args:
        query (str): The query string to search for. In English language, e.g., "What are the benefits of using AI in customer service?"

    Returns:
        list[Document]: A list of retrieved LangChain Document objects.
        
    Raises:
        ValueError: If the query is empty or contains only whitespace.
    """
    
    query = query.strip()
    query = query.replace("\n", " ")
    
    if not query:
        tools_logger.error("retrieve_documents_from_rag: empty query")
        raise ValueError("Query cannot be empty or whitespace.")
    
    tools_logger.info(f"Tool retrieve_documents_from_rag called - query: {query}")
    docs = HybridRetriever().retrieve(query, top_k=5)
    tools_logger.debug(f"Retrieved {len(docs)} documents: {[d.metadata.get('source') for d in docs]}")
    return docs

*Tools for human escalation by email and for the current context (location, time).*

In [ ]:
# source: src/agentic_customer_care/agent/tools/request_handler.py
import html
from datetime import datetime
from pydantic import BaseModel
from langchain_core.tools import tool 
import requests

tools_logger = setup_logger("tools")

@tool
def escalate_request_to_human(message: str) -> str:
    """
    Escalates a request to a human agent.
    In case the AI agent cannot handle a request or if the user explicitly requests human assistance, 
    this function can be called to escalate the request to a human agent for further handling.
    
    An email will be sent to the support team with the details of the request that needs human assistance.

    Args:
        message (str): The message or request that needs to be escalated.
        
    Returns:
        is_success (bool): True if the escalation was successful, False otherwise.
        error (str): Error message if the escalation failed, None otherwise.
    """
    
    tools_logger.info("Tool escalate_request_to_human called")
    tools_logger.debug(f"Escalation message: {message}")
    
    try:
    
        smtp_server = SmtpFactory.init_smtp_provider()
        smtp_server.send_email(
            to_email="g.monitillo.dev@gmail.com",
            subject="Escalated Request to Human Agent",
            body=f"Here is the details:\n\n{message}",
            html_body=ESCALATION_EMAIL_TEMPLATE.format(message=html.escape(message)),
        )
        
    except Exception as e:
        tools_logger.error(f"Failed to escalate request: {e}")
        return {"is_success": False, "error": str(e)}

    tools_logger.info("Request escalated to human agent")
    return {"is_success": True}

class Location(BaseModel):
    city: str
    timezone: str
    country:str
    region:str

class CurrentContext(BaseModel):
    location: Location
    date_time: str

@tool
def get_current_context() -> CurrentContext:
    """
    Retrieves the current context of the conversation or interaction.
    It returns a information object, which can be used to provide 
    additional information to the prompt like location and timestamp.

    Returns:
        location (Location): The location of the current context
        date_time (Date): The date and time of the current context
    """
    
    tools_logger.info("Tool get_current_context called")
    current_date_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        resp = requests.get("https://ipinfo.io/json")
        data = resp.json()
        location = Location(
            city=data.get("city"),
            timezone=data.get("timezone"),
            country=data.get("country"),
            region=data.get("region")
        )
    
    except Exception as e:
        tools_logger.error(f"Failed to retrieve current context: {e}")
        location = "Unknown"

    return CurrentContext(
        location=location,
        date_time=current_date_time
        )

*`get_all_tools`: the list of tools given to the agent.*

In [ ]:
# source: src/agentic_customer_care/agent/tools/tools_index.py
from langchain_core.tools import BaseTool


def get_all_tools() -> list[BaseTool]:
    """
    Get a list of all available tools.

    Returns:
        list: A list of tool functions.
    """
    
    return [
        retrieve_documents_from_rag,
        get_order_status,
        get_order_details,
        escalate_request_to_human,
        get_current_context    
    ]

Tools smoke test:

*Lists the tools and checks an order status.*

In [ ]:
print([t.name for t in get_all_tools()])
print(get_order_status.invoke({"order_id": "ORD-001"}))

## 2.4 Entity extractor

*`EntityExtractorAgent`: extracts user data and order info from the conversation.*

In [ ]:
# source: src/agentic_customer_care/agent/entity_extractor.py
import os

from dotenv import load_dotenv
import json

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,  AIMessage, ToolMessage



load_dotenv()

class ExtractorMessage(BaseModel):
    content: str
    source:str
class EntityExtractorAgent:
    
    def __init__(self):
        
        self.agent = None
        
        self.logger = setup_logger("agent", True)
            
        self.init_agent() # Start the initialization


    def init_agent(self):
        """ 
            Agent definition and initialization
        """
        
        model = init_chat_model(
            model=os.getenv("EXTRACTOR_LLM_MODEL","openai/gpt-oss-120b"),
            model_provider=os.getenv("EXTRACTOR_LLM_PROVIDER","openai"),
            temperature=0, 
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=os.getenv("OPENAI_API_BASE") or None
        )
        self.agent = create_agent(
            model=model,
            system_prompt=CC_AGENT_EXTRACTOR_PROMPT,
            response_format=ToolStrategy(SessionEntityData), # Define a tool useful to format the response in a given BaseModel
            debug=os.getenv("AGENT_VERBOSE", "false").lower() == "true",
        )
        


    def invoke(self, context: str) -> SessionEntityData | dict:
        """ 
            Agent invoke with context data.
            
            Args:
                context (str): conversation context
            
            Returns:
                SessionEntityData
        """
        
        messages = self._get_messages_from_context(context)
        if len(messages) == 0:
            return {}
        
        result = self.agent.invoke(self._build_input(messages))
        structured_response = result["structured_response"]
        
        self.logger.info(f"Entity extraction requested")
        self.logger.debug(f"Entity extracted -> {structured_response}")
        
        return structured_response

    @staticmethod
    def _build_input(messages: list[ExtractorMessage]) -> dict:
        serialized_messages = json.dumps([m.model_dump() for m in messages], indent=2, ensure_ascii=False)
        return {"messages": [{"role": "user", "content": serialized_messages}]}
    
    @staticmethod
    def _get_messages_from_context(context) -> list[ExtractorMessage]:
        
        messages = context["messages"]
                
        if not messages or (len(messages) <= 0):
            return []
                
        formatted_messages: list[ExtractorMessage] = []      
        for msg in messages:
                    
            formatted_messages.append( ExtractorMessage(
                content = msg.content or "",
                source  = type(msg).__name__
                ))
            
        return formatted_messages
    

def test_enity_extractor():
   
    EG_CONTEXT = {'messages': [HumanMessage(content='Ciao sono Nicola', additional_kwargs={}, response_metadata={}, id='727f82b3-57b0-4585-8e0c-ec3fb9c36fbb'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 872, 'total_tokens': 981, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 89, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.182525601, 'prompt_time': 0.048779949, 'completion_time': 0.11848283, 'total_time': 0.167262779}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ce54c53bd5', 'id': 'chatcmpl-7d4c7c63-b9cd-4276-8369-0aa496bd6cdf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a148f7-4dcd-7cd3-830b-bedf0d105259-0', tool_calls=[{'name': 'get_current_context', 'args': {}, 'id': 'fc_0ca26183-c162-4527-b686-6b6227eb4c5b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 872, 'output_tokens': 109, 'total_tokens': 981, 'input_token_details': {}, 'output_token_details': {'reasoning': 89}}), ToolMessage(content="location=Location(city='Andria', timezone='Europe/Rome', country='IT', region='Apulia') date_time='2026-10-17 10:25:29'", name='get_current_context', id='9f2516e8-724c-4f17-bb4c-7392514d17b1', tool_call_id='fc_0ca26183-c162-4527-b686-6b6227eb4c5b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 933, 'total_tokens': 1019, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.047226204, 'prompt_time': 0.083343906, 'completion_time': 0.088796056, 'total_time': 0.172139962}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'id': 'chatcmpl-4653213a-6a44-4c9f-8be0-5f4d5d69a662', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a148f7-50c2-7061-bdbf-3eafcb80daa8-0', tool_calls=[{'name': 'AgentResponse', 'args': {'answer': 'Ciao Nicola! Come posso aiutarti oggi con i nostri prodotti da giardino?', 'confidence': 9, 'needs_humans': False, 'sources': 'ConversationContext'}, 'id': 'fc_5d92451f-3314-4832-bb83-0a39be0ae887', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 933, 'output_tokens': 86, 'total_tokens': 1019, 'input_token_details': {}, 'output_token_details': {'reasoning': 31}}), ToolMessage(content="Returning structured response: answer='Ciao Nicola! Come posso aiutarti oggi con i nostri prodotti da giardino?' confidence=9.0 sources='ConversationContext' needs_humans=False", name='AgentResponse', id='9db025c8-1912-40a2-8093-cdca57cdcc76', tool_call_id='fc_5d92451f-3314-4832-bb83-0a39be0ae887')], 'structured_response': AgentResponse(answer='Ciao Nicola! Come posso aiutarti oggi con i nostri prodotti da giardino?', confidence=9.0, sources='ConversationContext', needs_humans=False)}
   
    agent = EntityExtractorAgent()
    
    result = agent.invoke(EG_CONTEXT)
    res = result.model_dump()
    print("🚀 ~ res:", res)

*Runs the extractor on a sample conversation.*

In [ ]:
test_enity_extractor()

## 2.5 Main workflow

*`CustomerCareAgent`: ReAct agent with tools, memory and session handling.*

In [ ]:
# source: src/agentic_customer_care/agent/workflow.py
import asyncio
import os
import uuid

from dotenv import load_dotenv

from langchain.agents import create_agent

from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, RemoveMessage, trim_messages
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain_core.messages.utils import count_tokens_approximately


load_dotenv()

MAX_TOOL_CALL_RETRIES = 1
RETRYABLE_PROVIDER_ERRORS = ("tool_use_failed", "output_parse_failed")


class CustomerCareAgent:
    
    def __init__(self):
        
        self.tools = get_all_tools() # Get all fuction / tools typed as StructuredTool that will be passed to the model

        self.logger = setup_logger("agent", True)
        self.agent = None
        
        self.session_data: dict[str, SessionEntity] = {} 
        self.entity_extr_agent = EntityExtractorAgent()
        
        self.kb_retriever = HybridRetriever() # Initialize singleton class of retriever (loads weights just one time at init operation)
        self.init_agent() # Start the initialization

    def init_agent(self):
        
        """ 
            Agent definition and initialization
        """
        model = init_chat_model(
            model=os.getenv("LLM_MODEL","openai/gpt-oss-120b"),
            model_provider=os.getenv("LLM_PROVIDER","openai"),
            temperature=0, 
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=os.getenv("OPENAI_API_BASE") or None
        )
        self.agent = create_agent(
            model=model,
            tools=self.tools,
            middleware=[build_prompt_enrichment(CC_AGENT_SYSTEM_PROMPT), build_entities_updater(self.entity_extr_agent)],
            context_schema=SessionEntity,
            response_format=ToolStrategy(AgentResponse), # Define a tool useful to format the response in a given BaseModel
            checkpointer=InMemorySaver(), # Checkpointer based on sessionId
            debug=os.getenv("AGENT_VERBOSE", "false").lower() == "true",
        )
        
        self.logger.info(f"Customer care Agent initialized - Model: {os.getenv("LLM_MODEL","openai/gpt-oss-120b")}")
        
        

    async def interact(self, sessionId: str, message: str) -> InteractResponse:
        """ 
            Agent invoke with session data (async).
            
            Args:
                sessionId (UUID)
                message (str): user query
            
            Returns:
                AgentResponse: Structured json response from agent
                sessionId: UUID used to store the historical context
            
            Entity extraction runs in the background after the answer: it is not awaited here.
        """

        sessionId = self._check_session(sessionId=sessionId)
        session_data: SessionEntity = self.session_data[sessionId]
        
        self.logger.info(f"New agent interaction for sessionId: {sessionId}")
        self.logger.debug(f"SessionData of {sessionId} -> {session_data}")
       
        self.checkpointer_config = {"configurable": {"thread_id": sessionId}}
        self._trim_messages()
        
        result = await self._invoke_with_retry(message, session_data)
        
        self.logger.debug(f"Response for {sessionId} -> {result}")
        
        return InteractResponse(
                data = result["structured_response"].model_dump(), 
                sessionId = sessionId
            )
        
    async def _invoke_with_retry(self, message: str, session_data: SessionEntity) -> dict:
        """
            Invoke the agent, retrying when the provider rejects a malformed / missing tool call
            (e.g. Groq 400 tool_use_failed). Retries resume from the checkpoint (input None), so the
            user message is not duplicated in the history.
            If the run finally fails the entity extraction hook would never run, so the entities
            of the user message are extracted anyway before re-raising.
        """
        
        agent_input = {"messages": [{"role": "user", "content": message}]}
        
        for attempt in range(MAX_TOOL_CALL_RETRIES + 1):
            try:
                return await self.agent.ainvoke(
                    agent_input,
                    config=self.checkpointer_config,
                    context=session_data,
                )
            except Exception as e:
                retryable = any(code in str(e) for code in RETRYABLE_PROVIDER_ERRORS)
                if retryable and attempt < MAX_TOOL_CALL_RETRIES:
                    self.logger.warning(f"Model call failed ({e}) - retry {attempt + 1}/{MAX_TOOL_CALL_RETRIES}")
                    agent_input = None  # resume from the last checkpoint
                    continue
                asyncio.create_task(self._extract_entities_after_failure(message, session_data))
                raise

    async def _extract_entities_after_failure(self, message: str, session_data: SessionEntity) -> None:
        try:
            history = self.agent.get_state(self.checkpointer_config).values.get("messages", [])
            extracted = await asyncio.to_thread(
                self.entity_extr_agent.invoke, {"messages": [*history, HumanMessage(content=message)]}
            )
            if extracted:
                session_data.merge_data(extracted)
                self.logger.debug(f"Session entities updated after failed run -> {extracted}")
        except Exception as e:
            self.logger.error(f"Entity extraction after failed run failed: {e}")

    def export_session_data(self, sessionId: str) -> SessionEntity:
        """ 
            Return data of a specific session.
        """
        
        if sessionId and sessionId in self.session_data:
            return self.session_data[sessionId]
        
        else:
            raise ValueError("Unknown sessionId")
        
    
    def _check_session(self, sessionId: str | None = None) -> str:
        
        """ 
            Return validated current sessionId or create new one if not exists
                
            Args:
                sessionId (str)
                    
            Returns:
                sessionId (str)
        """
            
        if sessionId and sessionId in self.session_data:
            return sessionId

        sessionId = str(uuid.uuid4())
        self.session_data[sessionId] = SessionEntity(sessionId=sessionId)
        return sessionId

    def _trim_messages(self):
               
        state = self.agent.get_state(self.checkpointer_config)
        history = state.values.get("messages", [])
        if history:
            trimmed = trim_messages(
                history,
                max_tokens=MAX_MEMORY_TOKEN, 
                strategy="last",
                token_counter=count_tokens_approximately,
                include_system=True, start_on="human",
            )
            if len(trimmed) < len(history):
                self.agent.update_state(
                    self.checkpointer_config,
                    {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *trimmed]},
            )


*Sends two messages to the agent and prints the collected session data.*

In [ ]:
import asyncio, nest_asyncio; nest_asyncio.apply()

agent = CustomerCareAgent()
r = await agent.interact(None, "What is the status of order 001?")
print(r.data)
r2 = await agent.interact(r.sessionId, "What is your return policy?")
print(r2.data)
await asyncio.sleep(5)  # let the background entity extraction finish
print(agent.export_session_data(r.sessionId))

# 3. API gateway

`init_gateway()` of the repo starts uvicorn with `reload=True` from an import string, which does not work inside a notebook.
It is replaced by `start_gateway()`, which serves the same `agent_gateway` app in a background thread.

*FastAPI app (`/health`, `/system_info`, `/inference`, `/session/{id}/data`) and `start_gateway`/`stop_gateway`.*

In [ ]:
# source: src/agentic_customer_care/api_gateway/fast_api.py
from fastapi import FastAPI, HTTPException, Response, Cookie
import os
import uvicorn

from pydantic import BaseModel

agent_logger = setup_logger("agent")

class InferenceRequest(BaseModel):
    message: str


agent_gateway = FastAPI()

cc_agent = CustomerCareAgent()

@agent_gateway.get("/health")
async def health():
    return {"status": "available"}

@agent_gateway.get('/system_info')
async def system_info():
    return {
        "version": "1.0.0",
        "author": "Giuseppe Monitillo",
        "description": "LLM Mock provider - This provider will return always 200 and proper responses",
        "status": "PartiallyAvailable"
    }
    
@agent_gateway.post('/inference')
async def chat_inference(
    payload: InferenceRequest,
    response: Response,
    sessionId = Cookie(None)
):
    
    try:
    
        interect_response = await cc_agent.interact(sessionId, payload.message)
  
        """ 
            Set sessionId cookie that will be used in next iterations 
        """  
        response.set_cookie(key="sessionId", value=interect_response.sessionId)

        return interect_response.data
    
    except Exception as e:
                
        error_message = getattr(e, "message", None) or str(e)
        status_code = getattr(e, "status_code", 500)
        agent_logger.error(f"Inference failed (sessionId: {sessionId}, status: {status_code}): {error_message}")
        raise HTTPException(status_code=status_code, detail=f"Sorry, something went wrong. - Details: {error_message}")
        
    
@agent_gateway.get('/session/{session_id}/data')
def get_session_data(session_id: str):
    
    try:
        return cc_agent.export_session_data(sessionId=session_id)
    
    except Exception as e:
        agent_logger.error(f"Failed to export session data (session_id: {session_id}): {e}")
        return e


    
import asyncio, threading, time

_gateway_server = None

def _serve():
    # own event loop: uvicorn's asyncio.run is patched by nest_asyncio and rejects loop_factory
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(_gateway_server.serve())

def _free_port(port: int):
    """Stop the in-notebook server and kill any other process listening on the port."""
    global _gateway_server
    if _gateway_server and not _gateway_server.should_exit:
        _gateway_server.should_exit = True
        time.sleep(1)  # let the previous server shut down
    try:
        import psutil
    except ImportError:
        !pip install -q psutil
        import psutil
    for conn in psutil.net_connections(kind="tcp"):
        if conn.laddr and conn.laddr.port == port and conn.status == psutil.CONN_LISTEN and conn.pid not in (None, os.getpid()):
            print(f"Killing process {conn.pid} listening on port {port}")
            psutil.Process(conn.pid).kill()

def start_gateway():
    """Serve agent_gateway in a background thread (notebook replacement of init_gateway)."""
    global _gateway_server
    host = os.getenv("AGENT_GATEWAY_HOST", "127.0.0.1")
    port = int(os.getenv("AGENT_GATEWAY_PORT", "8100"))
    _free_port(port)
    _gateway_server = uvicorn.Server(uvicorn.Config(agent_gateway, host=host, port=port, log_level="warning"))
    threading.Thread(target=_serve, daemon=True).start()
    for _ in range(150):  # wait up to 30s, so a failed bind does not hang the cell
        if _gateway_server.started:
            break
        time.sleep(0.2)
    else:
        raise RuntimeError("Gateway did not start: check the error above")
    print(f"Agent gateway running on http://{host}:{port}")

def stop_gateway():
    if _gateway_server:
        _gateway_server.should_exit = True

# 4. Run all steps

Everything above is already defined, so the steps are plain function calls
(these replace the repo's `poetry run ...` commands).

| Repo command | Notebook |
|---|---|
| `ingest` | `run_ingestion_pipeline()` (1.3) |
| `test_retrival` | 1.4 / `test_retrieve_documents_from_vectorstore()` |
| `evaluate_rag` | 1.6 / `evaluate_rag()` |
| `test_extractor` | `test_enity_extractor()` (2.4) |
| `run_agent_gateway` | `start_gateway()` (below) |

Start the gateway and call it:

*Starts the gateway in a background thread.*

In [ ]:
start_gateway()

Interactive chat against the gateway (commands: `history`, `clear`, `exit`). The `requests.Session` keeps the `sessionId` cookie, so the conversation continues across messages.

In [ ]:
import requests
import os
import pandas as pd
from IPython.display import display

# Global session: preserves the sessionId cookie across inference calls
session = requests.Session()

def map_history(history_list):
    """Converts the session history list into a pandas DataFrame and displays it without truncating."""
    if not history_list:
        print("\nNo history recorded yet.")
        return

    table_data = []
    for idx, turn in enumerate(history_list, 1):
        role = turn.get("role", "").upper()
        msg = turn.get("message", "")

        msg_text = ""
        metadata = "-"

        if role == "USER":
            msg_text = str(msg)
        elif role == "ASSISTANT":
            if isinstance(msg, dict):
                msg_text = msg.get("answer", str(msg))

                details = []
                if "confidence" in msg:
                    details.append(f"Confidence: {msg['confidence']}")
                if "sources" in msg:
                    details.append(f"Sources: {msg['sources']}")
                if "needs_humans" in msg:
                    details.append(f"Escalate: {msg['needs_humans']}")
                if "error" in msg:
                    details.append(f"Error: {msg['error']}")
                metadata = " | ".join(details) if details else "None"
            else:
                msg_text = str(msg)

        table_data.append({
            "Step": idx,
            "Role": role,
            "Message": msg_text,
            "Metadata Details": metadata
        })

    df = pd.DataFrame(table_data).set_index("Step")
    pd.set_option('display.max_colwidth', None)

    print("\n" + "="*40 + " SESSION CHAT HISTORY & METADATA " + "="*40 + "\n")
    display(df)
    print("\n" + "="*113 + "\n")

def execute_inference(query: str):
    host = os.getenv("AGENT_GATEWAY_HOST", "127.0.0.1")
    port = os.getenv("AGENT_GATEWAY_PORT", "8100")
    url = f"http://{host}:{port}/inference"

    payload = {
        "message": query
    }

    try:
        response = session.post(url, json=payload)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

history = []
print("=== INTERACTIVE AGENT CHAT SESSION ===")
print("Instructions: Type your query to start chatting.")
print("Special Commands: 'history' to display metadata table, 'clear' to reset chat, 'exit' to quit.")

while True:
    try:
        query = input("\nYou -> ").strip()
        if not query:
            continue

        if query.lower() == "exit":
            print("Exiting session...")
            break

        if query.lower() == "history":
            map_history(history)
            continue

        if query.lower() == "clear":
            history = []
            session.cookies.clear()
            print("Chat history and cookies successfully cleared!")
            continue

        history.append({
            "role": "user",
            "message": query
        })

        response = execute_inference(query)

        history.append({
            "role": "assistant",
            "message": response
        })

        if "answer" in response:
            print(f"\nAgent -> {response['answer']}")
            meta_info = []
            if "confidence" in response:
                meta_info.append(f"Confidence: {response['confidence']}")
            if "sources" in response and response['sources'] != 'None':
                meta_info.append(f"Sources: {response['sources']}")
            if response.get("needs_humans"):
                meta_info.append("⚠️ Escalation requested!")
            if meta_info:
                print(f"      [Metadata] | {' | '.join(meta_info)}")
        else:
            print(f"\nAgent -> Error: {response.get('error')}")

    except KeyboardInterrupt:
        print("\n Session interrupted by user.")
        break
    except Exception as e:
        print(f"\n Unexpected Error: {e}")

# Automatically show the history table on exit
map_history(history)

Entities collected for the session:

In [ ]:
import time; time.sleep(5)  # background entity extraction
host, port = os.getenv("AGENT_GATEWAY_HOST", "127.0.0.1"), os.getenv("AGENT_GATEWAY_PORT", "8100")
print(session.get(f"http://{host}:{port}/session/{session.cookies['sessionId']}/data").json())

*Stops the gateway.*

In [ ]:
stop_gateway()

# 5. Documentation

> The README below describes the original Poetry project; in this notebook the `poetry run` commands map to function calls (see section 4).

# 🤖 Agentic Customer Care

**AI-powered customer care agent for GreenThumb Marketplace — faster answers, happier gardeners.**

## Overview

GreenThumb Marketplace sells gardening supplies (seeds, bulbs, tools, pots, soil, fertilizers). The customer care team handles many heterogeneous requests — product questions, order tracking, seasonal advice, returns and refunds — with large volume spikes in spring and autumn.

This project is an AI agent that autonomously handles first-level requests and escalates to a human operator when needed. It uses a **ReAct reasoning loop** to choose a tool at each step, keeping every decision traceable.

## Key Features

- Conversational agent with autonomous reasoning (ReAct)
- Knowledge-base search via RAG (semantic + BM25) over a Chroma vector store
- Real-time order status and details lookup
- Escalation to human operators via email
- Short-term (trimming) and long-term (RAG) memory
- FastAPI gateway with cookie-based sessions
- RAGAS-based evaluation

## Architecture

The user sends a message and the agent enters a reasoning loop: it analyzes the request, picks a tool (knowledge-base search, order lookup, human escalation, context), reads the result and iterates if more information is needed before answering.

Tools, memory and session entities are described in [Architecture](docs/architecture.md).

## Getting Started

### Prerequisites

- Python `>=3.13,<4.0`
- [Poetry](https://python-poetry.org/) `>=2`
- An OpenAI-compatible LLM endpoint and API key
- (Optional) An SMTP account, e.g. Mailtrap, for human escalation emails

### Installation

```bash
git clone <repo-url>
cd Agentic-Customer-Care
poetry install
```

### Environment Setup

```bash
cp .env.example .env
```

Then edit `.env`:

| Variable | Default | Description |
|----------|---------|-------------|
| `OPENAI_API_KEY` | — | API key for the LLM (and embeddings if `EMBEDDING_PROVIDER=openai`). **Required.** |
| `OPENAI_API_BASE` | official OpenAI | Base URL of an OpenAI-compatible endpoint. |
| `LLM_PROVIDER` | `openai` | Provider of the main agent model. |
| `LLM_MODEL` | `openai/gpt-oss-120b` | Main agent model. |
| `EXTRACTOR_LLM_PROVIDER` | `openai` | Provider of the entity-extractor / evaluation model. |
| `EXTRACTOR_LLM_MODEL` | `openai/gpt-oss-120b` | Entity-extractor / evaluation model. |
| `EMBEDDING_PROVIDER` | `huggingface` | `huggingface` or `openai`. |
| `EMBEDDING_MODEL_NAME` | provider default | Embedding model name. |
| `SMTP_PROVIDER` | `mailtrap` | Email provider for escalations. |
| `SMTP_SERVER` / `SMTP_PORT` | `smtp.mailtrap.io` / `2525` | SMTP host and port. |
| `SMTP_USERNAME` / `SMTP_PASSWORD` | — | SMTP credentials. |
| `SMTP_SENDER` | `no-reply@yourdomain.com` | Sender address. |
| `AGENT_GATEWAY_HOST` / `AGENT_GATEWAY_PORT` | `127.0.0.1` / `8100` | API gateway bind address. |
| `AGENT_VERBOSE` | `false` | Print agent steps to stdout. |

### Commands

Scripts are defined in `pyproject.toml` and run with `poetry run`:

| Command | Description |
|---------|-------------|
| `poetry run ingest` | Chunk and index `knowledge_base/documents` into the vector store. |
| `poetry run run_agent_gateway` | Start the FastAPI gateway (default `http://127.0.0.1:8100`). |
| `poetry run test_retrival` | Smoke-test retrieval from the vector store. |
| `poetry run test_extractor` | Smoke-test the entity extractor. |
| `poetry run evaluate_rag` | Evaluate retrieval with RAGAS. Prompts for `top_k` and `semantic_ratio` (0–1, 1 = fully semantic), then prints the metrics. Uses the `EXTRACTOR_LLM_*` model. |

### Run

```bash
poetry run ingest               # first time only
poetry run run_agent_gateway
```

Try it:

```bash
curl -c cookies.txt -b cookies.txt -X POST http://127.0.0.1:8100/inference \
  -H "Content-Type: application/json" \
  -d '{"message": "What is the status of order ORD-001?"}'
```

The `sessionId` cookie keeps the conversation going across requests.

### API Endpoints

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/health` | Health check. |
| `GET` | `/system_info` | Service information. |
| `POST` | `/inference` | Send `{"message": "..."}`; returns the agent response and sets `sessionId` cookie. |
| `GET` | `/session/{session_id}/data` | Export entities collected for a session. |

Interactive docs are available at `/docs`.

## Project Structure

```
.
├── src/
│   ├── agentic_customer_care/
│   │   ├── agent/          # workflow, tools, middlewares, entity extractor, sessions
│   │   ├── api_gateway/    # FastAPI app
│   │   ├── providers/      # embeddings, SMTP
│   │   └── rag/            # ingestion, retrieval, evaluation
│   └── utils/              # constants, logger, helpers
├── knowledge_base/         # documents, orders, products, ingestion chunks, vector store
├── docs/
├── tests/
├── .env.example
└── pyproject.toml
```

## Data & Mock Assets

Production data cannot be shared, so the project ships **fictional but realistic mock data**:

- **Knowledge base** — 30+ documents: growing guides, FAQ, policies, known platform errors.
- **Orders** — `knowledge_base/orders/orders_v1.json`, mock orders across several statuses.
- **Products** — `knowledge_base/products/catalog_v1.json`.

## Evaluation

The agent is evaluated on a dedicated dataset (`src/agentic_customer_care/rag/evaluation/`) with RAGAS to measure retrieval quality and answer faithfulness. Run it with `poetry run evaluate_rag`.

> 📊 Methodology, dataset and results: [Evaluation](docs/evaluation.md)

## Open Points

- **User data management and GDPR compliance.** As a POC, orders and products are read from the knowledge base, and the order tools have direct access to them. In production this would let users retrieve data through the agent, so authentication and authorization must be improved to return only the data each user is entitled to.

## Documentation

- [Architecture](docs/architecture.md) — flow, tools, memory and session entities
- [Evaluation](docs/evaluation.md) — evaluator, test dataset, test runs
- [Design Decisions](docs/decisions.md) — technical decisions and rationale

# Architecture

Rationale behind each choice is logged in [decisions.md](decisions.md).

## Request flow

1. The client calls `POST /inference` (FastAPI, `api_gateway/fast_api.py`); the `sessionId` cookie identifies the conversation.
2. `CustomerCareAgent.interact` (`agent/workflow.py`) resolves or creates the session, trims the history and runs the agent with `ainvoke`.
3. The agent loops (ReAct): it picks a tool, reads the result, and iterates until it can answer.
4. The response is returned and, in the background, the entity extractor updates the session entities.

## Tools

All tools are registered in `src/agentic_customer_care/agent/tools/tools_index.py`.

| Tool | Module | Purpose |
|------|--------|---------|
| `retrieve_documents_from_rag` | `tools/rag.py` | Searches the knowledge base (product sheets, growing guides, policies) with semantic + keyword (BM25) retrieval, grounding answers in verified company information. |
| `get_order_status` | `tools/oms.py` | Returns only the status of an order by ID (format `ORD-XXX`). |
| `get_order_details` | `tools/oms.py` | Returns full order information (status, shipping, estimated delivery) from the orders database. |
| `escalate_request_to_human` | `tools/request_handler.py` | Sends an email to the support team (SMTP) when confidence is low, the request is out of scope, or the customer asks for a human. |
| `get_current_context` | `tools/request_handler.py` | Provides extra context such as user region/city and current datetime (e.g. the current month, without asking the user). |

Tools can read session entities through `runtime.context`.

## Memory

The agent has three memory layers.

| Layer | Mechanism | Scope | Persistence |
|-------|-----------|-------|-------------|
| Short-term | Message history in a LangGraph `InMemorySaver`, trimmed | per `sessionId` | in process |
| Entity memory | `SessionEntity` in `CustomerCareAgent.session_data` | per `sessionId` | in process |
| Long-term | RAG over `knowledge_base/` (Chroma) | global | on disk |

Everything in process is lost on restart and not shared across instances.

### Short-term memory: trimming (D-002)

- The checkpointer holds only `messages`, keyed by `thread_id = sessionId`.
- Before each turn `_trim_messages` applies `trim_messages` (strategy `last`, system message kept, starts on a human message) with `MAX_MEMORY_TOKEN` (`src/utils/constants.py`, default `2000`) and approximate token counting.
- When the limit is exceeded the oldest messages are dropped, never rewritten.
- **Summarization is intentionally not used.** The domain (AMS-like support) has short contexts with few, critical values; an LLM summary is non-deterministic and could alter data (e.g. a wrong digit in an order id). Exact values stay verbatim in the history or in the entity memory.
- Because key user data lives in the entity memory, `MAX_MEMORY_TOKEN` can be lowered: shorter context helps the model stay focused.

### Entity memory (D-001)

Deterministic, schema-driven store (`SessionEntity`, typed fields such as `UserInfo`, `AddressInfo`) for the key data of a session: name, surname, email, address. It lives in `CustomerCareAgent.session_data`, keyed by `sessionId`, and is the single source of truth for entities (the checkpointer holds only `messages`).

- **Write:** after each answer, `EntityExtractorAgent` runs in the background (`@after_agent` middleware, `asyncio.create_task` + `to_thread`) and deep-merges its result via `merge_data`. A new non-`None` value overwrites the old one; `None` never erases. The main agent never writes entities.
- **Read:** a `@dynamic_prompt` middleware rebuilds the system prompt on every model call as base prompt + known entities, passed via `context_schema`. Entities are never stored in the checkpoint, so they don't accumulate and are unaffected by trimming.
- **Trade-offs:** one-turn lag (turn N sees data up to N-1); the agent must run with `ainvoke`; background errors are not propagated; entities are sent to the LLM.
- **Open points:** correcting or retracting a wrong value, entity validation, persistence/expiry, and keeping the extractor limited to stable fields.

### Long-term memory

A RAG layer over `knowledge_base/documents` (topics: `faq`, `garden_info`, `platform_errors`, `policies`) gives the agent product information, policies and agronomic guides across sessions, independently of the current conversation. It is global (not per session) and persisted on disk.

**Ingestion** (`poetry run ingest`, `rag/ingestion.py`):

1. **Extract:** read every file under `knowledge_base/documents/<topic>/` into a `Document` with metadata `topic` and `source`.
2. **Preprocess:** Markdown → HTML → plain text (`markdown2` + BeautifulSoup), whitespace collapsed, headings and tags removed.
3. **Chunk:** `RecursiveCharacterTextSplitter` with `chunk_size=1000`, `chunk_overlap=200`, separators `["\n\n", "\n", " ", ""]`. Overlap preserves context across chunk boundaries. Chunks keep the document metadata and are saved to `knowledge_base/ingestion/chunks.jsonl`.
4. **Embed and index:** chunks are embedded with the provider set by `EMBEDDING_PROVIDER` (`huggingface` or `openai`) and persisted in Chroma at `knowledge_base/vectorstore`. The script asks whether to clear the existing collection first.

**Retrieval** (`rag/retrival.py`, `HybridRetriever`): hybrid search with a LangChain `EnsembleRetriever` that merges two retrievers with weighted Reciprocal Rank Fusion:

| Retriever | Source | Strength | Default weight |
|-----------|--------|----------|----------------|
| Semantic | Chroma vector store | meaning, paraphrases | `semantic_ratio = 0.6` |
| BM25 | `chunks.jsonl` | exact keywords (ids, product names, error codes) | `1 - semantic_ratio = 0.4` |

- Each retriever returns its top `k` chunks; the ensemble fuses the rankings and keeps the first `top_k` (default `5`).
- `semantic_ratio` must be in `[0, 1]`, otherwise a `ValueError` is raised.
- `HybridRetriever` is a thread-safe singleton: embeddings, Chroma and chunks load once and are reused, reducing time to first token. The BM25 index is rebuilt from the chunks on each query.
- The `retrieve_documents_from_rag` tool strips the query, rejects empty ones and calls `retrieve(query, top_k=5)`. Queries must be in English, since the knowledge base is in English.
- Re-run `poetry run ingest` after changing documents: both the vector store and `chunks.jsonl` must be rebuilt so the two retrievers stay aligned.

# Design Decisions

Log of technical decisions and why they were taken. Newest entries at the bottom. Implementation details are in [architecture.md](architecture.md).

## D-001 — Session entities: dedicated memory injected via dynamic prompt

**Need.** The main agent must know user data (name, surname, email, address) on every turn without re-asking.

**Decision.** Keep entities in a typed `SessionEntity` per session, filled by a background extractor and injected into the system prompt through `context_schema` + `@dynamic_prompt`.

**Why.** It fits this use case: relevant information is saved once and reused to solve problems without asking the user again, even after old messages are trimmed.

**Technical choices**

- Extracting in the background after the answer keeps latency unaffected.
- A schema-driven store is deterministic and bounded in size; exact values are never rewritten by an LLM.
- Per-invoke context is not saved in the checkpoint: the prompt is rebuilt each call, so it never accumulates, never goes stale and is not affected by trimming.
- A single source of truth (`session_data`) avoids duplicating entities in the checkpoint.

**Alternatives rejected**

| Option | Why not |
|---|---|
| System message in `messages` | Persisted in the checkpoint, accumulates each turn, can go stale. |
| `state_schema` | Duplicates `session_data` as source of truth. |
| Tool with `ToolRuntime` | The model decides whether to call it, so it is unreliable. |
| `store` | Only useful to share data across threads/sessions. |

**Accepted trade-offs.** One-turn lag in entity availability; async-only; background errors not propagated; entities are sent to the LLM. Open points (corrections, validation, persistence) are listed in the architecture doc.

## D-002 — Short-term memory: trimming, not summarization

**Decision.** Keep trimming as the short-term memory strategy; do not introduce summarization.

**Why.** The agent works in an AMS-like context: little text, few but critical values (ids, names, emails).

1. Contexts are usually small, so compression gains little.
2. Summarization is non-deterministic and may alter data (e.g. a wrong digit in an order id) and mislead the conversation.

Dropping the oldest messages keeps exact values verbatim, and key user data is preserved anyway by the entity memory.

## D-003 — Long-term memory: deterministic RAG over a topic-organized KB

**Decision.** Long-term memory is a RAG over the knowledge base, organized by topic, using recursive chunking and a hybrid (semantic + BM25) retriever, with no LLM involved in retrieval.

**Why.** The agent must ground answers in verified company information, and the retrieved data must be predictable and reproducible.

**Knowledge base by topic.** Documents are kept in one folder per topic, stored as `topic` metadata on every chunk:

- `faq`
- `garden_info`
- `platform_errors`
- `policies`

**Chunking strategy.** `RecursiveCharacterTextSplitter`, chunks of 1000 characters with 200 overlap, splitting on paragraphs, then lines, then words. Documents are short and self-contained, so this keeps paragraphs together where possible, and the overlap avoids losing context at chunk boundaries. Markdown is converted to plain text first, so formatting noise doesn't enter the embeddings.

**No reranking or LLM augmentation.** Query rewriting, LLM reranking and similar techniques are intentionally not used: they would introduce non-deterministic behavior in KB retrieval, as in D-002. The same query always returns the same chunks, which makes retrieval easy to test and evaluate.

**Hybrid retrieval (semantic + BM25).** The KB contains platform errors (codes, exact messages), where semantic search alone can miss exact matches. An ensemble retriever combines BM25 (keyword) with vector search (meaning), weighted 40/60 by default, so both kinds of query work. Both approaches are deterministic.

# Evaluation

Evaluation of the RAG (long-term memory) with [RAGAS](https://docs.ragas.io). Retrieval design is described in [architecture.md](architecture.md) and the reasoning in [decisions.md](decisions.md) (D-003).

## Evaluator

Code: `src/agentic_customer_care/rag/evaluation/ragas_evauation.py`. Run with:

```bash
poetry run evaluate_rag
```

It asks for `top_k` and `semantic_ratio` (0–1, 1 = fully semantic) and then:

1. For each test case, retrieves chunks with `HybridRetriever` using the given parameters.
2. Generates an answer with a stuff-documents chain: the LLM must answer **only** from the retrieved documents and say so if the information is missing (temperature 0).
3. Builds a RAGAS dataset (`question`, `answer`, retrieved contexts, `ground_truth`) and scores it.
4. Prints models, retrieval parameters, number of cases and metrics.

Only retrieval + grounded generation is evaluated; the agent loop, tools and memory are not part of this pipeline.

**Models.** The same LLM is used for answer generation and as RAGAS judge (`EXTRACTOR_LLM_PROVIDER` / `EXTRACTOR_LLM_MODEL`). Embeddings come from `EMBEDDING_PROVIDER` / `EMBEDDING_MODEL_NAME`.

### Metrics

| Metric | What it measures |
|--------|------------------|
| `faithfulness` | Whether the answer is supported by the retrieved contexts (low = hallucination). |
| `answer_relevancy` | Whether the answer addresses the question. |
| `context_recall` | Whether the retrieved contexts contain the information of the ground truth (retrieval quality). |

## Test dataset

`eval_dataset_25.py` (`EVAL_DATASET_25`, the one used by the evaluator) contains 25 hand-written cases, each with a `question` and a reference `ground_truth`, spread across the KB topics:

| Topic | Cases |
|-------|-------|
| `faq` | 3 |
| `policies` | 5 |
| `platform_errors` | 3 |
| `garden_info` | 14 |

`eval_dataset.py` holds an extended set of 50 cases with the same topics; it is not used by default.

## Test runs

### Run 1 — fully semantic, large chunks

| Parameter | Value |
|-----------|-------|
| Chunk size | 2000 tokens |
| `top_k` | 5 |
| `semantic_ratio` | 1.0 (semantic only, BM25 disabled) |
| Generation / judge model | `openai/gpt-oss-20b` |
| Test cases | 25 |

| Metric | Score |
|--------|-------|
| Faithfulness | 0.8519 |
| Answer relevancy | 0.8399 |
| Context recall | 1.0000 |

**Reading.** Context recall is perfect: with large chunks the relevant information is always retrieved. Faithfulness and relevancy are lower, probably because large chunks carry a lot of unrelated text that distracts the model.

### Run 2 — hybrid retrieval (50/50)

| Parameter | Value |
|-----------|-------|
| Chunk size | 2000 tokens |
| `top_k` | 5 |
| `semantic_ratio` | 0.5 (50% semantic, 50% BM25) |
| Extractor model | `openai/gpt-oss-20b` |
| Test cases | 25 |

| Metric | Score |
|--------|-------|
| Faithfulness | 0.8805 |
| Answer relevancy | 0.8140 |
| Context recall | 0.9167 |

**Reading.** Compared with Run 1, faithfulness improves (0.8519 → 0.8805): BM25 plus semantic search surfaces more precise chunks, so answers rely better on the retrieved context. Context recall drops (1.0000 → 0.9167) and answer relevancy falls slightly (0.8399 → 0.8140). Without the fully semantic ranking, some relevant chunks are missed, and keyword matches may add less useful context. The run changes several things at once (retrieval mode, and possibly chunking), so the differences cannot be attributed to a single parameter.

### Colab notes

- **Python version**: the repo requires Python ≥ 3.13; the code also runs on Colab's 3.12.
- **Inline code**: cells share one namespace, so intra-project imports were removed and the per-module `logger` variables renamed (`rag_logger`, `tools_logger`, `agent_logger`); log files are still `log/<rag|tools|agent>/`.
- **Escalation emails** need valid SMTP credentials (Mailtrap); the recipient is hard-coded in `request_handler.py`.
- **Known issue fixed**: `test_enity_extractor` in the repo indexes `result["structured_response"]`, but the extractor already returns the entity data; the notebook uses `result.model_dump()`.
- **Persistence**: `/content` is wiped when the runtime is recycled; copy `knowledge_base/vectorstore` to Drive to keep it.